In [1]:
"""
============================================================
BANGLADESH CYCLONE TRACK PREDICTION
Complete Multi-Horizon Pipeline — Thesis Grade
============================================================
STANDARDIZED GLOBAL HYPERPARAMETERS (hard-coded, never changed):
  GLOBAL_LR    = 0.01
  GLOBAL_EPOCHS = 300

Deep Learning  : AdamW, LR=0.01, Epochs=300, Dropout=0.2,
                 Activation=ReLU, Batch=128, EarlyStopping=OFF
Boosting       : learning_rate=0.01, iterations/n_estimators=300
RandomForest   : n_estimators=300
Linear/Ridge/  : SGDRegressor with eta0=0.01, max_iter=300
Lasso

Custom models  : same budget, multi-horizon joint loss
============================================================
"""

# ─────────────────────────────────────────────────────────────
# 0. IMPORTS
# ─────────────────────────────────────────────────────────────
import os, warnings, time, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import entropy as scipy_entropy

warnings.filterwarnings("ignore")
random.seed(42); np.random.seed(42)

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, \
                             HistGradientBoostingRegressor, StackingRegressor
from sklearn.linear_model import SGDRegressor, Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Dense, LSTM, GRU, Bidirectional,
                                     Conv1D, Flatten, Dropout, LayerNormalization,
                                     MultiHeadAttention, GlobalAveragePooling1D,
                                     Add, BatchNormalization, Lambda, Concatenate)
from tensorflow.keras.optimizers import AdamW
import tensorflow.keras.backend as K
tf.random.set_seed(42)

# ─────────────────────────────────────────────────────────────
# GLOBAL HYPERPARAMETERS — NEVER CHANGE THESE
# ─────────────────────────────────────────────────────────────
GLOBAL_LR      = 0.01
GLOBAL_EPOCHS  = 300
GLOBAL_BATCH   = 128
GLOBAL_DROPOUT = 0.2
GLOBAL_ACT     = "relu"
DL_OPT         = lambda: AdamW(learning_rate=GLOBAL_LR)

# Fixed tree params
RF_P   = dict(n_estimators=300, max_depth=None, n_jobs=-1, random_state=42)
XGB_P  = dict(n_estimators=300, learning_rate=GLOBAL_LR, max_depth=6,
              subsample=0.8, colsample_bytree=0.8,
              random_state=42, tree_method="hist", verbosity=0)
LGB_P  = dict(n_estimators=300, learning_rate=GLOBAL_LR, max_depth=6,
              num_leaves=63, random_state=42, verbose=-1)
CB_P   = dict(iterations=300, learning_rate=GLOBAL_LR, depth=6,
              loss_function="MultiRMSE", random_seed=42, verbose=0)
CB_P1  = dict(iterations=300, learning_rate=GLOBAL_LR, depth=6,
              random_seed=42, verbose=0)   # single target CatBoost

# SGD params for linear models
SGD_P  = dict(max_iter=300, eta0=GLOBAL_LR, learning_rate="constant",
              tol=None, random_state=42)

SEQ_LEN   = 4
HORIZONS  = {"3h": 1, "12h": 4, "24h": 8, "48h": 16}
H_LABELS  = ["3h", "12h", "24h", "48h"]
H_STEPS   = [1, 4, 8, 16]
H_HOURS   = [3, 12, 24, 48]
W_LOSS    = [1.0, 1.0, 1.0, 1.0]   # joint loss weights, equal to start

# ─────────────────────────────────────────────────────────────
# DIRECTORIES
# ─────────────────────────────────────────────────────────────
for d in ["outputs/plots","outputs/metrics","outputs/predictions"]:
    os.makedirs(d, exist_ok=True)

# ─────────────────────────────────────────────────────────────
# HAVERSINE
# ─────────────────────────────────────────────────────────────
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    la1,lo1,la2,lo2 = map(np.radians,[lat1,lon1,lat2,lon2])
    a = np.sin((la2-la1)/2)**2 + np.cos(la1)*np.cos(la2)*np.sin((lo2-lo1)/2)**2
    return R * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

# ─────────────────────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────────────────────
print("\n"+"="*60)
print("1. LOADING & CLEANING")
print("="*60)
DATA_PATH = "/kaggle/input/datasets/anybodyk/bbddddd/bangladesh_nextstep_dataset_research_ready.csv"
df_raw = pd.read_csv(DATA_PATH)
df_raw = df_raw[df_raw["flag_latnext_mismatch"] == 0]
df_raw = df_raw[df_raw["flag_irregular_dt"]     == 0]
df_raw = df_raw.dropna(subset=["dLAT","dLON","LAT_next","LON_next"])
df_raw = df_raw.reset_index(drop=True)
print(f"  Rows: {len(df_raw):,}   Storms: {df_raw['SID'].nunique()}")


2026-03-05 04:31:20.372999: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772685080.705982     106 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772685080.792587     106 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772685081.538629     106 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772685081.538659     106 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772685081.538662     106 computation_placer.cc:177] computation placer alr


1. LOADING & CLEANING
  Rows: 46,053   Storms: 1519


In [2]:
# ─────────────────────────────────────────────────────────────
# 2. PHYSICS FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────
print("\n[2] Physics Feature Engineering …")

df = df_raw.copy()
df["ISO_TIME"] = pd.to_datetime(df["ISO_TIME"], errors="coerce")
doy = df["ISO_TIME"].dt.dayofyear.fillna(180)
df["season_sin"]          = np.sin(2*np.pi*doy/365)
df["season_cos"]          = np.cos(2*np.pi*doy/365)
df["curvature"]           = df.groupby("SID")["geo_bearing"].diff().fillna(0).abs()
df["bearing_stability"]   = 1.0 / (1.0 + df.groupby("SID")["geo_bearing"]
                                .transform(lambda x: x.rolling(3,min_periods=1).std().fillna(0)))
df["speed_stability"]     = 1.0 / (1.0 + df.groupby("SID")["geo_speed_kmh"]
                                .transform(lambda x: x.rolling(3,min_periods=1).std().fillna(0)))
df["bearing_consistency"] = (df.groupby("SID")["geo_bear_sin"]
                               .transform(lambda x: x.rolling(3,min_periods=1).std().fillna(0)) +
                              df.groupby("SID")["geo_bear_cos"]
                               .transform(lambda x: x.rolling(3,min_periods=1).std().fillna(0)))
df["dlat_accel"]  = df["dLAT_lag1"] - df["dLAT_lag2"].fillna(df["dLAT_lag1"])
df["dlon_accel"]  = df["dLON_lag1"] - df["dLON_lag2"].fillna(df["dLON_lag1"])
df["dist_BoB"]    = haversine_km(df["LAT"].values, df["LON"].values, 15.0, 88.0)
df["speed_sq"]    = df["geo_speed_kmh"] ** 2
df["lat_lon_int"] = df["LAT"] * df["LON"]


# ── Long-horizon trend features (key for 48h prediction) ──
# 3-step velocity trend — captures acceleration/deceleration
df["dlat_trend"] = (df.groupby("SID")["dLAT_lag1"]
                    .transform(lambda x: x.rolling(3,min_periods=1)
                    .apply(lambda v: (v.iloc[-1]-v.iloc[0]) if len(v)>1 else 0, raw=False)
                    .fillna(0)))
df["dlon_trend"] = (df.groupby("SID")["dLON_lag1"]
                    .transform(lambda x: x.rolling(3,min_periods=1)
                    .apply(lambda v: (v.iloc[-1]-v.iloc[0]) if len(v)>1 else 0, raw=False)
                    .fillna(0)))
# Bearing momentum — persistence of turning direction
df["bearing_momentum"] = df.groupby("SID")["geo_bearing"].diff().fillna(0)
# Speed trend — is storm accelerating or decelerating?
df["speed_trend"] = (df.groupby("SID")["geo_speed_kmh"]
                     .transform(lambda x: x.rolling(3,min_periods=1).mean().fillna(0))
                     - df["geo_speed_kmh"].fillna(0))
# Recurvature proxy — lat above 20 with eastward component
df["recurvature_proxy"] = ((df["LAT"] > 20).astype(float) *
                            (df["geo_bear_cos"] > 0).astype(float))
# Distance to land change rate
df["dist2land_rate"] = df.groupby("SID")["DIST2LAND"].diff().fillna(0)

ALL_FEATURE_COLS = [c for c in [
    "LAT","LON","lat_lon_int",
    "LAT_lag1","LON_lag1","dLAT_lag1","dLON_lag1",
    "LAT_lag2","LON_lag2","dLAT_lag2","dLON_lag2",
    "LAT_lag3","LON_lag3","dLAT_lag3","dLON_lag3",
    "geo_speed_kmh","speed_sq","geo_bearing",
    "geo_bear_sin","geo_bear_cos",
    "DIST2LAND","LANDFALL","STORM_SPEED","STORM_DIR","NEWDELHI_WIND",
    "LAT_lag2_missing","LON_lag2_missing","dLAT_lag2_missing","dLON_lag2_missing",
    "LAT_lag3_missing","LON_lag3_missing","dLAT_lag3_missing","dLON_lag3_missing",
    "NEWDELHI_WIND_missing",
    "curvature","bearing_stability","speed_stability","bearing_consistency",
    "season_sin","season_cos","dlat_accel","dlon_accel","dist_BoB",
    "dlat_trend","dlon_trend","bearing_momentum","speed_trend",
    "recurvature_proxy","dist2land_rate",
] if c in df.columns]

print(f"  Total features: {len(ALL_FEATURE_COLS)}")

# ─────────────────────────────────────────────────────────────
# 3. BUILD MULTI-HORIZON TARGETS (within-SID, no leakage)
# ─────────────────────────────────────────────────────────────
print("\n[3] Building multi-horizon targets …")
t0 = time.time()
records = []
for sid, grp in df.groupby("SID"):
    grp = grp.reset_index(drop=True)
    n   = len(grp)
    max_k = max(H_STEPS)
    for t in range(n - max_k):
        row = grp.iloc[t].to_dict()
        valid = True
        for label, k in zip(H_LABELS, H_STEPS):
            fut = grp.iloc[t + k]
            row[f"LAT_{label}"]  = fut["LAT"]
            row[f"LON_{label}"]  = fut["LON"]
            row[f"dLAT_{label}"] = fut["LAT"] - grp.iloc[t]["LAT"]
            row[f"dLON_{label}"] = fut["LON"] - grp.iloc[t]["LON"]
            if pd.isna(row[f"LAT_{label}"]): valid = False
        if valid: records.append(row)

mh_df = pd.DataFrame(records)
print(f"  Multi-horizon dataset: {len(mh_df):,} rows  [{time.time()-t0:.1f}s]")

# ─────────────────────────────────────────────────────────────
# 4. STORM-WISE SPLIT (same seed everywhere)
# ─────────────────────────────────────────────────────────────
print("\n[4] Storm-wise split …")
storms = mh_df["SID"].unique()
np.random.seed(42); np.random.shuffle(storms)
n = len(storms); n_tr = int(n*0.70); n_vl = int(n*0.15)
train_storms = set(storms[:n_tr])
val_storms   = set(storms[n_tr:n_tr+n_vl])
test_storms  = set(storms[n_tr+n_vl:])

mh_train = mh_df[mh_df["SID"].isin(train_storms)].copy()
mh_val   = mh_df[mh_df["SID"].isin(val_storms)].copy()
mh_test  = mh_df[mh_df["SID"].isin(test_storms)].copy()
print(f"  Train:{len(mh_train):,}  Val:{len(mh_val):,}  Test:{len(mh_test):,}")

# 3h-only frames (for baselines — more data, only need 1-step target)
df_tr3 = df[df["SID"].isin(train_storms)].copy()
df_vl3 = df[df["SID"].isin(val_storms)].copy()
df_ts3 = df[df["SID"].isin(test_storms)].copy()

# ─────────────────────────────────────────────────────────────
# 5. PREPROCESSING (fit on train only)
# ─────────────────────────────────────────────────────────────
print("\n[5] Preprocessing …")
imputer = SimpleImputer(strategy="median")
scaler  = StandardScaler()

# 3h preprocessor
X3_tr = scaler.fit_transform(imputer.fit_transform(df_tr3[ALL_FEATURE_COLS].values))
X3_vl = scaler.transform(imputer.transform(df_vl3[ALL_FEATURE_COLS].values))
X3_ts = scaler.transform(imputer.transform(df_ts3[ALL_FEATURE_COLS].values))
y3_tr = df_tr3[["dLAT","dLON"]].values
y3_vl = df_vl3[["dLAT","dLON"]].values
y3_ts = df_ts3[["dLAT","dLON"]].values
lat3_ts = df_ts3["LAT"].values; lon3_ts = df_ts3["LON"].values

# Multi-horizon preprocessor (same scaler — fit already done)
Xmh_tr = scaler.transform(imputer.transform(mh_train[ALL_FEATURE_COLS].values))
Xmh_vl = scaler.transform(imputer.transform(mh_val[ALL_FEATURE_COLS].values))
Xmh_ts = scaler.transform(imputer.transform(mh_test[ALL_FEATURE_COLS].values))
latmh_ts = mh_test["LAT"].values; lonmh_ts = mh_test["LON"].values

# All horizon targets stacked for multi-horizon models
def get_mh_targets(frame):
    return [frame[[f"dLAT_{h}",f"dLON_{h}"]].values for h in H_LABELS]

ymh_tr = get_mh_targets(mh_train)   # list of 4 arrays, each (n,2)
ymh_vl = get_mh_targets(mh_val)
ymh_ts = get_mh_targets(mh_test)

n_feat = X3_tr.shape[1]
print(f"  Feature dims: {n_feat}")

# ─────────────────────────────────────────────────────────────
# METRICS HELPER
# ─────────────────────────────────────────────────────────────
def compute_metrics(lc, ln, y_true, y_pred, label):
    errors = haversine_km(lc+y_true[:,0], ln+y_true[:,1],
                          lc+y_pred[:,0], ln+y_pred[:,1])
    return {
        "model"    : label,
        "median_km": round(float(np.median(errors)), 3),
        "mean_km"  : round(float(np.mean(errors)),   3),
        "p75_km"   : round(float(np.percentile(errors, 75)), 3),
        "p90_km"   : round(float(np.percentile(errors, 90)), 3),
        "rmse_dlat": round(float(np.sqrt(mean_squared_error(y_true[:,0], y_pred[:,0]))), 5),
        "rmse_dlon": round(float(np.sqrt(mean_squared_error(y_true[:,1], y_pred[:,1]))), 5),
        "r2_dlat"  : round(float(r2_score(y_true[:,0], y_pred[:,0])), 5),
        "r2_dlon"  : round(float(r2_score(y_true[:,1], y_pred[:,1])), 5),
        "errors"   : errors
    }

# Storage
all_results   = {h: {} for h in H_LABELS}   # horizon → model → metrics
all_preds     = {h: {} for h in H_LABELS}   # horizon → model → y_pred
dl_history    = {}                           # model_h → history
sece_base_preds = {}                         # for catropy

# ─────────────────────────────────────────────────────────────
# DL SEQUENCE BUILDER
# ─────────────────────────────────────────────────────────────
def make_seq(X, y_list, seq_len=SEQ_LEN):
    """Returns Xs (n,seq,feat) and ys (list of n,2 arrays)"""
    n = len(X)
    Xs = []; ys = [[] for _ in y_list]
    for i in range(seq_len-1, n):
        Xs.append(X[i-seq_len+1:i+1])
        for j, y in enumerate(y_list):
            ys[j].append(y[i])
    return (np.array(Xs, dtype=np.float32),
            [np.array(yj, dtype=np.float32) for yj in ys])

Xseq_tr, yseq_tr = make_seq(Xmh_tr, ymh_tr)
Xseq_vl, yseq_vl = make_seq(Xmh_vl, ymh_vl)
Xseq_ts, yseq_ts = make_seq(Xmh_ts, ymh_ts)
lat_sq = latmh_ts[SEQ_LEN-1:]; lon_sq = lonmh_ts[SEQ_LEN-1:]


[2] Physics Feature Engineering …
  Total features: 49

[3] Building multi-horizon targets …
  Multi-horizon dataset: 24,287 rows  [17.3s]

[4] Storm-wise split …
  Train:16,712  Val:3,879  Test:3,696

[5] Preprocessing …
  Feature dims: 49


In [3]:
# ─────────────────────────────────────────────────────────────
# 6. BASELINE CLASSICAL MODELS (3h + direct horizon)
# ─────────────────────────────────────────────────────────────
print("\n"+"="*60)
print("6. BASELINE CLASSICAL MODELS")
print("="*60)

D = GLOBAL_DROPOUT; ACT = GLOBAL_ACT

def sgd_wrap(penalty=None):
    """MultiOutput SGD regressor matching the global LR/epoch budget."""
    base = SGDRegressor(penalty=penalty, alpha=0.0001, eta0=GLOBAL_LR,
                        learning_rate="constant", max_iter=GLOBAL_EPOCHS,
                        tol=None, random_state=42)
    return MultiOutputRegressor(base)

def rec_ml_3h(name, y_pred):
    m = compute_metrics(lat3_ts, lon3_ts, y3_ts, y_pred, name)
    all_results["3h"][name] = m; all_preds["3h"][name] = y_pred
    print(f"  {name:<35} Med={m['median_km']:.2f}  R²lat={m['r2_dlat']:.3f}")

def rec_ml_horizon(name, h_label, lc, ln, y_true, y_pred):
    m = compute_metrics(lc, ln, y_true, y_pred, name)
    all_results[h_label][name] = m; all_preds[h_label][name] = y_pred

# Persistence baseline — 3h
persist_3h = np.column_stack([df_ts3["dLAT_lag1"].fillna(0).values,
                               df_ts3["dLON_lag1"].fillna(0).values])
rec_ml_3h("Persistence Baseline", persist_3h)

# Persistence baseline — rollout horizons
for h_label, k_steps in zip(H_LABELS[1:], H_STEPS[1:]):
    persist_h = np.column_stack([mh_test["dLAT_lag1"].fillna(0).values,
                                  mh_test["dLON_lag1"].fillna(0).values])
    y_true_h  = mh_test[[f"dLAT_{h_label}", f"dLON_{h_label}"]].values
    rec_ml_horizon("Persistence Baseline", h_label, latmh_ts, lonmh_ts, y_true_h, persist_h)

# Classical models — train once on 3h, evaluate on 3h
# Also train on MH data for each rollout horizon
classical_specs = [
    ("Linear (SGD)",        sgd_wrap(None)),
    ("Ridge (SGD)",         sgd_wrap("l2")),
    ("Lasso (SGD)",         sgd_wrap("l1")),
    ("Random Forest",       RandomForestRegressor(**RF_P)),
    ("Gradient Boosting",   MultiOutputRegressor(GradientBoostingRegressor(
                                n_estimators=300, learning_rate=GLOBAL_LR,
                                max_depth=5, random_state=42))),
    ("HistGradientBoosting",MultiOutputRegressor(HistGradientBoostingRegressor(
                                max_iter=300, learning_rate=GLOBAL_LR,
                                max_depth=6, random_state=42))),
    ("XGBoost",             MultiOutputRegressor(xgb.XGBRegressor(**XGB_P))),
    ("LightGBM",            MultiOutputRegressor(lgb.LGBMRegressor(**LGB_P))),
    ("CatBoost",            CatBoostRegressor(**CB_P)),
]

print("\n  [3h baselines]")
for name, mdl in classical_specs:
    if name == "CatBoost":
        mdl.fit(X3_tr, y3_tr, eval_set=(X3_vl, y3_vl))
    else:
        mdl.fit(X3_tr, y3_tr)
    rec_ml_3h(name, mdl.predict(X3_ts))

# Direct horizon models (re-train per horizon)
print("\n  [Rollout direct baselines]")
for h_label in H_LABELS[1:]:
    print(f"\n    — Horizon {h_label} —")
    y_tr_h = mh_train[[f"dLAT_{h_label}",f"dLON_{h_label}"]].values
    y_vl_h = mh_val[[f"dLAT_{h_label}",f"dLON_{h_label}"]].values
    y_ts_h = mh_test[[f"dLAT_{h_label}",f"dLON_{h_label}"]].values
    for name, _ in classical_specs:
        # Build fresh model each horizon
        if   name == "Linear (SGD)":         mdl2 = sgd_wrap(None)
        elif name == "Ridge (SGD)":          mdl2 = sgd_wrap("l2")
        elif name == "Lasso (SGD)":          mdl2 = sgd_wrap("l1")
        elif name == "Random Forest":        mdl2 = RandomForestRegressor(**RF_P)
        elif name == "Gradient Boosting":    mdl2 = MultiOutputRegressor(GradientBoostingRegressor(
                                                 n_estimators=300,learning_rate=GLOBAL_LR,max_depth=5,random_state=42))
        elif name == "HistGradientBoosting": mdl2 = MultiOutputRegressor(HistGradientBoostingRegressor(
                                                 max_iter=300,learning_rate=GLOBAL_LR,max_depth=6,random_state=42))
        elif name == "XGBoost":              mdl2 = MultiOutputRegressor(xgb.XGBRegressor(**XGB_P))
        elif name == "LightGBM":             mdl2 = MultiOutputRegressor(lgb.LGBMRegressor(**LGB_P))
        elif name == "CatBoost":             mdl2 = CatBoostRegressor(**CB_P)
        else: continue

        if name == "CatBoost":
            mdl2.fit(Xmh_tr, y_tr_h, eval_set=(Xmh_vl, y_vl_h))
        else:
            mdl2.fit(Xmh_tr, y_tr_h)
        p = mdl2.predict(Xmh_ts)
        rec_ml_horizon(name, h_label, latmh_ts, lonmh_ts, y_ts_h, p)
        print(f"      {name:<30} Med={all_results[h_label][name]['median_km']:.2f}")

# Stacking Ensemble — 3h + multi-horizon
# Also store val predictions for use as SECE base learner
print("\n  [Stacking Ensemble 3h]")
base_est = [
    ("rf",  RandomForestRegressor(n_estimators=150,n_jobs=-1,random_state=42)),
    ("xgb", xgb.XGBRegressor(n_estimators=150,learning_rate=GLOBAL_LR,max_depth=6,
                               random_state=42,tree_method="hist",verbosity=0)),
    ("lgb", lgb.LGBMRegressor(n_estimators=150,learning_rate=GLOBAL_LR,random_state=42,verbose=-1)),
]

# Storage for val+test predictions (used later as SECE base learner)
stk_val_preds  = {}
stk_test_preds = {}

# 3h
sl=StackingRegressor(estimators=base_est,final_estimator=Ridge(),cv=3,n_jobs=-1)
sn=StackingRegressor(estimators=base_est,final_estimator=Ridge(),cv=3,n_jobs=-1)
sl.fit(X3_tr,y3_tr[:,0]); sn.fit(X3_tr,y3_tr[:,1])
stack_pred = np.column_stack([sl.predict(X3_ts),sn.predict(X3_ts)])
rec_ml_3h("Stacking Ensemble", stack_pred)
# Store val preds aligned to mh_val for SECE
stk_val_preds["3h"]  = np.column_stack([sl.predict(Xmh_vl), sn.predict(Xmh_vl)])
stk_test_preds["3h"] = np.column_stack([sl.predict(Xmh_ts), sn.predict(Xmh_ts)])

for h_label in H_LABELS[1:]:
    y_tr_h=mh_train[[f"dLAT_{h_label}",f"dLON_{h_label}"]].values
    y_ts_h=mh_test[[f"dLAT_{h_label}",f"dLON_{h_label}"]].values
    y_vl_h=mh_val[[f"dLAT_{h_label}",f"dLON_{h_label}"]].values
    sl2=StackingRegressor(estimators=base_est,final_estimator=Ridge(),cv=3,n_jobs=-1)
    sn2=StackingRegressor(estimators=base_est,final_estimator=Ridge(),cv=3,n_jobs=-1)
    sl2.fit(Xmh_tr,y_tr_h[:,0]); sn2.fit(Xmh_tr,y_tr_h[:,1])
    p=np.column_stack([sl2.predict(Xmh_ts),sn2.predict(Xmh_ts)])
    rec_ml_horizon("Stacking Ensemble",h_label,latmh_ts,lonmh_ts,y_ts_h,p)
    stk_val_preds[h_label]  = np.column_stack([sl2.predict(Xmh_vl),sn2.predict(Xmh_vl)])
    stk_test_preds[h_label] = p

print("\n  Classical + Stacking baselines done.")

# ─────────────────────────────────────────────────────────────
# 7. DL BASELINE MODELS (multi-horizon outputs, joint MSE)
# EarlyStopping = OFF as required
# ─────────────────────────────────────────────────────────────
print("\n"+"="*60)
print("7. DEEP LEARNING BASELINES  [AdamW|LR=0.01|ReLU|Drop=0.2|Ep=300|BS=128|NoES]")
print("="*60)

def build_multi_out_head(x, name_prefix):
    """Attach 4 output heads for 3h/12h/24h/48h."""
    outs = []
    for h in H_LABELS:
        o = Dense(16, activation=ACT)(x)
        o = Dense(2, name=f"{name_prefix}_{h}")(o)
        outs.append(o)
    return outs

def joint_loss(y_true, y_pred):
    """MSE summed over lat/lon — used as per-horizon loss."""
    return K.mean(K.square(y_true - y_pred))

def compile_multi(model):
    model.compile(
        optimizer=DL_OPT(),
        loss={f"out_{h}": joint_loss for h in H_LABELS},
        loss_weights={f"out_{h}": w for h, w in zip(H_LABELS, W_LOSS)}
    )

# Storage for best DL model predictions (used as SECE base learners)
dl_sece_val_preds  = {}   # {model_name: {h_label: (n_seq_val, 2)}}
dl_sece_test_preds = {}   # {model_name: {h_label: (n_seq_ts, 2)}}
SECE_DL_MODELS = ["BLSTM", "CNN-GRU"]  # Best DL models to feed into SECE

def fit_dl_multi(model, name):
    """Fit multi-output DL model, log history, evaluate all horizons.
    Also stores val+test predictions for BLSTM and CNN-GRU to use in SECE.
    """
    y_dict_tr = {f"out_{h}": yseq_tr[i] for i, h in enumerate(H_LABELS)}
    y_dict_vl = {f"out_{h}": yseq_vl[i] for i, h in enumerate(H_LABELS)}
    hist = model.fit(
        Xseq_tr, y_dict_tr,
        validation_data=(Xseq_vl, y_dict_vl),
        epochs=GLOBAL_EPOCHS,
        batch_size=GLOBAL_BATCH,
        verbose=0,
        callbacks=[]    # EarlyStopping OFF
    )
    dl_history[name] = hist.history
    preds    = model.predict(Xseq_ts, verbose=0)
    preds_vl = model.predict(Xseq_vl, verbose=0)

    # Keras returns dict when outputs are named dicts, list/tuple otherwise
    def _parse(p):
        if isinstance(p, dict):    return p
        elif isinstance(p, (list, tuple)):
            return {f"out_{h}": p[i] for i, h in enumerate(H_LABELS)}
        else: return {f"out_{H_LABELS[0]}": p}

    preds_by_h    = _parse(preds)
    preds_vl_by_h = _parse(preds_vl)

    for i, h in enumerate(H_LABELS):
        key = f"out_{h}"
        if key in preds_by_h:
            rec_ml_horizon(name, h, lat_sq, lon_sq, yseq_ts[i], preds_by_h[key])

    # Store val+test preds for best DL models → used as SECE base learners
    if name in SECE_DL_MODELS:
        dl_sece_val_preds[name]  = {}
        dl_sece_test_preds[name] = {}
        for h in H_LABELS:
            key = f"out_{h}"
            if key in preds_by_h:
                dl_sece_val_preds[name][h]  = preds_vl_by_h[key]   # (n_seq_vl, 2)
                dl_sece_test_preds[name][h] = preds_by_h[key]       # (n_seq_ts, 2)

    print(f"    {name}")
    for h in H_LABELS:
        if name in all_results[h]:
            m = all_results[h][name]
            print(f"      {h}: Med={m['median_km']:.2f}  R2lat={m['r2_dlat']:.3f}")
        else:
            print(f"      {h}: (no result — output key mismatch)")

def out_names():
    return {f"out_{h}": f"out_{h}" for h in H_LABELS}

# ── LSTM ──
print("\n  ▶ LSTM")
def mk_lstm():
    i = Input((SEQ_LEN, n_feat))
    x = LSTM(128, return_sequences=True)(i); x = Dropout(D)(x)
    x = LSTM(64)(x);                         x = Dropout(D)(x)
    x = Dense(64, activation=ACT)(x)
    outs = build_multi_out_head(x, "out")
    m = Model(i, {f"out_{h}": o for h, o in zip(H_LABELS, outs)})
    compile_multi(m); return m
fit_dl_multi(mk_lstm(), "LSTM")

# ── GRU ──
print("\n  ▶ GRU")
def mk_gru():
    i = Input((SEQ_LEN, n_feat))
    x = GRU(128, return_sequences=True)(i);  x = Dropout(D)(x)
    x = GRU(64)(x);                          x = Dropout(D)(x)
    x = Dense(64, activation=ACT)(x)
    outs = build_multi_out_head(x, "out")
    m = Model(i, {f"out_{h}": o for h, o in zip(H_LABELS, outs)})
    compile_multi(m); return m
fit_dl_multi(mk_gru(), "GRU")

# ── BLSTM ──
print("\n  ▶ BLSTM")
def mk_blstm():
    i = Input((SEQ_LEN, n_feat))
    x = Bidirectional(LSTM(128, return_sequences=True))(i); x = Dropout(D)(x)
    x = Bidirectional(LSTM(64))(x);                        x = Dropout(D)(x)
    x = Dense(64, activation=ACT)(x)
    outs = build_multi_out_head(x, "out")
    m = Model(i, {f"out_{h}": o for h, o in zip(H_LABELS, outs)})
    compile_multi(m); return m
fit_dl_multi(mk_blstm(), "BLSTM")

# ── SLSTM ──
print("\n  ▶ SLSTM")
def mk_slstm():
    i = Input((SEQ_LEN, n_feat))
    x = LSTM(128, return_sequences=True)(i)
    x = LSTM(96,  return_sequences=True)(x)
    x = LSTM(64,  return_sequences=True)(x)
    x = LSTM(32)(x); x = Dropout(D)(x)
    x = Dense(64, activation=ACT)(x)
    outs = build_multi_out_head(x, "out")
    m = Model(i, {f"out_{h}": o for h, o in zip(H_LABELS, outs)})
    compile_multi(m); return m
fit_dl_multi(mk_slstm(), "SLSTM")

# ── CNN ──
print("\n  ▶ CNN")
def mk_cnn():
    i = Input((SEQ_LEN, n_feat))
    x = Conv1D(128, 2, activation=ACT, padding="same")(i)
    x = Conv1D(64,  2, activation=ACT, padding="same")(x)
    x = Flatten()(x); x = Dense(64, activation=ACT)(x); x = Dropout(D)(x)
    outs = build_multi_out_head(x, "out")
    m = Model(i, {f"out_{h}": o for h, o in zip(H_LABELS, outs)})
    compile_multi(m); return m
fit_dl_multi(mk_cnn(), "CNN")

# ── CNN-GRU ──
print("\n  ▶ CNN-GRU")
def mk_cnngru():
    i = Input((SEQ_LEN, n_feat))
    x = Conv1D(64, 2, activation=ACT, padding="same")(i)
    x = GRU(64)(x); x = Dropout(D)(x)
    x = Dense(64, activation=ACT)(x)
    outs = build_multi_out_head(x, "out")
    m = Model(i, {f"out_{h}": o for h, o in zip(H_LABELS, outs)})
    compile_multi(m); return m
fit_dl_multi(mk_cnngru(), "CNN-GRU")

# ── CNN-LSTM ──
print("\n  ▶ CNN-LSTM")
def mk_cnnlstm():
    i = Input((SEQ_LEN, n_feat))
    x = Conv1D(64, 2, activation=ACT, padding="same")(i)
    x = LSTM(64)(x); x = Dropout(D)(x)
    x = Dense(64, activation=ACT)(x)
    outs = build_multi_out_head(x, "out")
    m = Model(i, {f"out_{h}": o for h, o in zip(H_LABELS, outs)})
    compile_multi(m); return m
fit_dl_multi(mk_cnnlstm(), "CNN-LSTM")

# ── GRU-LSTM ──
print("\n  ▶ GRU-LSTM")
def mk_grulstm():
    i = Input((SEQ_LEN, n_feat))
    x = GRU(64, return_sequences=True)(i)
    x = LSTM(64)(x); x = Dropout(D)(x)
    x = Dense(64, activation=ACT)(x)
    outs = build_multi_out_head(x, "out")
    m = Model(i, {f"out_{h}": o for h, o in zip(H_LABELS, outs)})
    compile_multi(m); return m
fit_dl_multi(mk_grulstm(), "GRU-LSTM")

# ── Hybrid Transformer ──
print("\n  ▶ Hybrid Transformer")
def mk_transformer():
    i = Input((SEQ_LEN, n_feat))
    x = Conv1D(128, 1, activation=ACT)(i)
    x = LayerNormalization()(x)
    attn, _ = MultiHeadAttention(num_heads=4, key_dim=32)(x, x, return_attention_scores=True)
    x = Add()([x, attn]); x = LayerNormalization()(x)
    x = GRU(64, return_sequences=True)(x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation=ACT)(x); x = Dropout(D)(x)
    outs = build_multi_out_head(x, "out")
    m = Model(i, {f"out_{h}": o for h, o in zip(H_LABELS, outs)})
    compile_multi(m); return m
fit_dl_multi(mk_transformer(), "Hybrid Transformer")

print("\n  DL baselines complete.")


6. BASELINE CLASSICAL MODELS
  Persistence Baseline                Med=11.12  R²lat=0.723

  [3h baselines]
  Linear (SGD)                        Med=9894.10  R²lat=-3211856814553933635649536.000
  Ridge (SGD)                         Med=9900.48  R²lat=-2990336451320075228545024.000
  Lasso (SGD)                         Med=9968.52  R²lat=-3316542815654804101005312.000
  Random Forest                       Med=4.83  R²lat=0.924
  Gradient Boosting                   Med=6.48  R²lat=0.886
  HistGradientBoosting                Med=6.30  R²lat=0.893
  XGBoost                             Med=6.06  R²lat=0.900
  LightGBM                            Med=5.81  R²lat=0.909
  CatBoost                            Med=8.05  R²lat=0.845

  [Rollout direct baselines]

    — Horizon 12h —
      Linear (SGD)                   Med=10031.12
      Ridge (SGD)                    Med=9902.71
      Lasso (SGD)                    Med=10027.55
      Random Forest                  Med=30.18
      Gradient Boost

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

  Stacking Ensemble                   Med=4.64  R²lat=0.930


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v


  Classical + Stacking baselines done.

7. DEEP LEARNING BASELINES  [AdamW|LR=0.01|ReLU|Drop=0.2|Ep=300|BS=128|NoES]

  ▶ LSTM


I0000 00:00:1772686210.530222     106 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 70896 MB memory:  -> device: 0, name: NVIDIA H100 80GB HBM3, pci bus id: 0000:04:00.0, compute capability: 9.0
E0000 00:00:1772686211.574435    3146 ptx_compiler_helpers.cc:88] *** WARNING *** Invoking ptxas with version 12.5.82, which corresponds to a CUDA version <=12.6.2. CUDA versions 12.x.y up to and including 12.6.2 miscompile certain edge cases around clamping.
Please upgrade to CUDA 12.6.3 or newer.
I0000 00:00:1772686215.440008    3142 cuda_dnn.cc:529] Loaded cuDNN version 91002


    LSTM
      3h: Med=12.83  R2lat=0.486
      12h: Med=52.33  R2lat=0.429
      24h: Med=114.83  R2lat=0.329
      48h: Med=257.66  R2lat=0.191

  ▶ GRU
    GRU
      3h: Med=13.14  R2lat=0.493
      12h: Med=49.91  R2lat=0.484
      24h: Med=110.40  R2lat=0.391
      48h: Med=248.43  R2lat=0.278

  ▶ BLSTM
    BLSTM
      3h: Med=12.56  R2lat=0.481
      12h: Med=48.57  R2lat=0.449
      24h: Med=112.98  R2lat=0.289
      48h: Med=265.08  R2lat=0.117

  ▶ SLSTM
    SLSTM
      3h: Med=14.24  R2lat=0.418
      12h: Med=53.84  R2lat=0.400
      24h: Med=118.00  R2lat=0.313
      48h: Med=275.26  R2lat=0.168

  ▶ CNN


I0000 00:00:1772687719.038468    3144 service.cc:152] XLA service 0x7b92d80a74f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1772687719.038498    3144 service.cc:160]   StreamExecutor device (0): NVIDIA H100 80GB HBM3, Compute Capability 9.0
I0000 00:00:1772687725.616047    3144 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
E0000 00:00:1772687728.073798    3135 buffer_comparator.cc:156] Difference at 3: 0.513641, expected 0.351403
E0000 00:00:1772687728.073840    3135 buffer_comparator.cc:156] Difference at 19: 0.54541, expected 0.378069
2026-03-05 05:15:28.073850: E external/local_xla/xla/service/gpu/autotuning/gemm_fusion_autotuner.cc:1138] Results do not match the reference. This is likely a bug/unexpected loss of precision.


    CNN
      3h: Med=11.17  R2lat=0.509
      12h: Med=45.77  R2lat=0.490
      24h: Med=108.53  R2lat=0.362
      48h: Med=258.76  R2lat=0.234

  ▶ CNN-GRU
    CNN-GRU
      3h: Med=11.35  R2lat=0.583
      12h: Med=44.77  R2lat=0.555
      24h: Med=102.17  R2lat=0.462
      48h: Med=238.58  R2lat=0.340

  ▶ CNN-LSTM
    CNN-LSTM
      3h: Med=11.62  R2lat=0.559
      12h: Med=45.95  R2lat=0.518
      24h: Med=106.55  R2lat=0.404
      48h: Med=246.47  R2lat=0.278

  ▶ GRU-LSTM
    GRU-LSTM
      3h: Med=12.93  R2lat=0.488
      12h: Med=51.94  R2lat=0.434
      24h: Med=116.15  R2lat=0.323
      48h: Med=263.12  R2lat=0.211

  ▶ Hybrid Transformer
    Hybrid Transformer
      3h: Med=13.89  R2lat=0.380
      12h: Med=56.55  R2lat=0.334
      24h: Med=124.22  R2lat=0.277
      48h: Med=270.41  R2lat=0.237

  DL baselines complete.


In [4]:
# ─────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────
# 8. CUSTOM MODELS — HORIZON-CONSISTENT WINNERS
# ─────────────────────────────────────────────────────────────
print("\n"+"="*60)
print("8. CUSTOM MODELS")
print("="*60)

# ── Self-contained setup (safe to run standalone or after Sections 1-7) ──
import os, warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge, SGDRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Dense, LSTM, GRU, Conv1D, Dropout,
                                      BatchNormalization, Add, LayerNormalization)
from tensorflow.keras.optimizers import AdamW

# Global constants
GLOBAL_LR      = 0.01
GLOBAL_EPOCHS  = 300
GLOBAL_BATCH   = 128
GLOBAL_DROPOUT = 0.2
GLOBAL_ACT     = "relu"
D   = GLOBAL_DROPOUT
ACT = GLOBAL_ACT
DEG_TO_KM_LAT = 111.32
DEG_TO_KM_LON = 103.50
SEQ_LEN  = 4
H_LABELS = ["3h", "12h", "24h", "48h"]
H_STEPS  = [1, 4, 8, 16]
H_HOURS  = [3, 12, 24, 48]
W_LOSS   = [1.0, 1.0, 1.0, 1.0]

CB_P  = dict(iterations=300, learning_rate=GLOBAL_LR, depth=6,
             loss_function="MultiRMSE", random_seed=42, verbose=0)
XGB_P = dict(n_estimators=300, learning_rate=GLOBAL_LR, max_depth=6,
             subsample=0.8, colsample_bytree=0.8,
             random_state=42, tree_method="hist", verbosity=0)
LGB_P = dict(n_estimators=300, learning_rate=GLOBAL_LR, max_depth=6,
             num_leaves=63, random_state=42, verbose=-1)

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    la1,lo1,la2,lo2 = map(np.radians,[lat1,lon1,lat2,lon2])
    a = np.sin((la2-la1)/2)**2 + np.cos(la1)*np.cos(la2)*np.sin((lo2-lo1)/2)**2
    return R * 2 * np.arcsin(np.sqrt(np.clip(a,0,1)))

def compute_metrics(lc, ln, y_true, y_pred, label):
    errors = haversine_km(lc+y_true[:,0], ln+y_true[:,1],
                          lc+y_pred[:,0],  ln+y_pred[:,1])
    return {
        "model"    : label,
        "median_km": round(float(np.median(errors)),3),
        "mean_km"  : round(float(np.mean(errors)),3),
        "p75_km"   : round(float(np.percentile(errors,75)),3),
        "p90_km"   : round(float(np.percentile(errors,90)),3),
        "rmse_dlat": round(float(np.sqrt(mean_squared_error(y_true[:,0],y_pred[:,0]))),5),
        "rmse_dlon": round(float(np.sqrt(mean_squared_error(y_true[:,1],y_pred[:,1]))),5),
        "r2_dlat"  : round(float(r2_score(y_true[:,0],y_pred[:,0])),5),
        "r2_dlon"  : round(float(r2_score(y_true[:,1],y_pred[:,1])),5),
        "errors"   : errors
    }

# Check if variables already exist from Sections 1-7
_need_rebuild = "mh_train" not in dir() or mh_train is None

if _need_rebuild:
    print("  [Rebuilding data from scratch — Sections 1-7 not in scope]")
    np.random.seed(42)

    DATA_PATH = "/kaggle/input/datasets/anybodyk/bbddddd/bangladesh_nextstep_dataset_research_ready.csv"
    df_raw = pd.read_csv(DATA_PATH)
    df_raw = df_raw[df_raw["flag_latnext_mismatch"]==0]
    df_raw = df_raw[df_raw["flag_irregular_dt"]==0]
    df_raw = df_raw.dropna(subset=["dLAT","dLON","LAT_next","LON_next"])
    df_raw = df_raw.reset_index(drop=True)
    df = df_raw.copy()
    df["ISO_TIME"] = pd.to_datetime(df["ISO_TIME"], errors="coerce")
    doy = df["ISO_TIME"].dt.dayofyear.fillna(180)
    df["season_sin"]        = np.sin(2*np.pi*doy/365)
    df["season_cos"]        = np.cos(2*np.pi*doy/365)
    df["curvature"]         = df.groupby("SID")["geo_bearing"].diff().fillna(0).abs()
    df["bearing_stability"] = 1.0/(1.0+df.groupby("SID")["geo_bearing"]
                                .transform(lambda x: x.rolling(3,min_periods=1).std().fillna(0)))
    df["speed_stability"]   = 1.0/(1.0+df.groupby("SID")["geo_speed_kmh"]
                                .transform(lambda x: x.rolling(3,min_periods=1).std().fillna(0)))
    df["bearing_consistency"]=(df.groupby("SID")["geo_bear_sin"]
                               .transform(lambda x: x.rolling(3,min_periods=1).std().fillna(0))+
                               df.groupby("SID")["geo_bear_cos"]
                               .transform(lambda x: x.rolling(3,min_periods=1).std().fillna(0)))
    df["dlat_accel"] = df["dLAT_lag1"]-df["dLAT_lag2"].fillna(df["dLAT_lag1"])
    df["dlon_accel"] = df["dLON_lag1"]-df["dLON_lag2"].fillna(df["dLON_lag1"])
    df["dist_BoB"]   = haversine_km(df["LAT"].values,df["LON"].values,15.0,88.0)
    df["speed_sq"]   = df["geo_speed_kmh"]**2
    df["lat_lon_int"]= df["LAT"]*df["LON"]

    ALL_FEATURE_COLS = [c for c in [
        "LAT","LON","lat_lon_int",
        "LAT_lag1","LON_lag1","dLAT_lag1","dLON_lag1",
        "LAT_lag2","LON_lag2","dLAT_lag2","dLON_lag2",
        "LAT_lag3","LON_lag3","dLAT_lag3","dLON_lag3",
        "geo_speed_kmh","speed_sq","geo_bearing",
        "geo_bear_sin","geo_bear_cos",
        "DIST2LAND","LANDFALL","STORM_SPEED","STORM_DIR","NEWDELHI_WIND",
        "LAT_lag2_missing","LON_lag2_missing","dLAT_lag2_missing","dLON_lag2_missing",
        "LAT_lag3_missing","LON_lag3_missing","dLAT_lag3_missing","dLON_lag3_missing",
        "NEWDELHI_WIND_missing",
        "curvature","bearing_stability","speed_stability","bearing_consistency",
        "season_sin","season_cos","dlat_accel","dlon_accel","dist_BoB",
    ] if c in df.columns]

    # Build multi-horizon targets
    records = []
    for sid, grp in df.groupby("SID"):
        grp = grp.reset_index(drop=True); n = len(grp)
        for t in range(n - max(H_STEPS)):
            row = grp.iloc[t].to_dict(); valid = True
            for label, k in zip(H_LABELS, H_STEPS):
                fut = grp.iloc[t+k]
                row[f"LAT_{label}"]  = fut["LAT"]; row[f"LON_{label}"]  = fut["LON"]
                row[f"dLAT_{label}"] = fut["LAT"]-grp.iloc[t]["LAT"]
                row[f"dLON_{label}"] = fut["LON"]-grp.iloc[t]["LON"]
                if pd.isna(row[f"LAT_{label}"]): valid = False
            if valid: records.append(row)
    mh_df = pd.DataFrame(records)

    # Storm-wise split
    storms = mh_df["SID"].unique()
    np.random.shuffle(storms)
    n=len(storms); n_tr=int(n*0.70); n_vl=int(n*0.15)
    train_storms=set(storms[:n_tr]); val_storms=set(storms[n_tr:n_tr+n_vl])
    test_storms =set(storms[n_tr+n_vl:])
    mh_train=mh_df[mh_df["SID"].isin(train_storms)].copy()
    mh_val  =mh_df[mh_df["SID"].isin(val_storms)].copy()
    mh_test =mh_df[mh_df["SID"].isin(test_storms)].copy()
    df_tr3=df[df["SID"].isin(train_storms)].copy()
    df_vl3=df[df["SID"].isin(val_storms)].copy()
    df_ts3=df[df["SID"].isin(test_storms)].copy()

    # Preprocessing
    imputer=SimpleImputer(strategy="median"); scaler=StandardScaler()
    X3_tr=scaler.fit_transform(imputer.fit_transform(df_tr3[ALL_FEATURE_COLS].values))
    X3_vl=scaler.transform(imputer.transform(df_vl3[ALL_FEATURE_COLS].values))
    X3_ts=scaler.transform(imputer.transform(df_ts3[ALL_FEATURE_COLS].values))
    y3_tr=df_tr3[["dLAT","dLON"]].values
    y3_vl=df_vl3[["dLAT","dLON"]].values
    y3_ts=df_ts3[["dLAT","dLON"]].values
    lat3_ts=df_ts3["LAT"].values; lon3_ts=df_ts3["LON"].values

    Xmh_tr=scaler.transform(imputer.transform(mh_train[ALL_FEATURE_COLS].values))
    Xmh_vl=scaler.transform(imputer.transform(mh_val[ALL_FEATURE_COLS].values))
    Xmh_ts=scaler.transform(imputer.transform(mh_test[ALL_FEATURE_COLS].values))
    latmh_ts=mh_test["LAT"].values; lonmh_ts=mh_test["LON"].values

    persist_3h=np.column_stack([df_ts3["dLAT_lag1"].fillna(0).values,
                                  df_ts3["dLON_lag1"].fillna(0).values])
    n_feat=X3_tr.shape[1]

    # all_results and all_preds dicts (empty — only custom models populated here)
    all_results={h:{} for h in H_LABELS}
    all_preds  ={h:{} for h in H_LABELS}
    print(f"  Data rebuilt: mh_train={len(mh_train):,}  mh_val={len(mh_val):,}  mh_test={len(mh_test):,}")
else:
    print("  Variables found from Sections 1-7 — skipping rebuild")

def rec_ml_horizon(name, h_label, lc, ln, y_true, y_pred):
    m = compute_metrics(lc, ln, y_true, y_pred, name)
    all_results[h_label][name]=m; all_preds[h_label][name]=y_pred

# ─────────────────────────────────────────────────────────────
DEG_TO_KM_LAT = 111.32
DEG_TO_KM_LON = 103.50

# ── 8A. CatBoost + MotionNN (bias correction layer) ──────────
# ── 8A. CatBoost + MotionNN (bias correction layer) ──────────
# Architecture:
#   Step 1: Train CatBoost on all horizons (already done in Section 6/7
#           but we retrain here on mh data for clean isolation)
#   Step 2: For each horizon, convert CatBoost error to km space
#   Step 3: Train a small NN to predict the correction vector (Δlat_km, Δlon_km)
#           using: CatBoost predicted dx/dy + storm motion features
#   Step 4: Final prediction = CatBoost + NN correction (back in degrees)
#   Version: separate correction model per horizon (train_one_lead)
#            with motion features (speed, bearing, curvature, accel)
print("\n  ▶ CB+MotionNN (CatBoost + per-horizon bias correction NN)")

# --- Step 1: Train CatBoost base per horizon ---
cb_base_preds_tr = {}   # horizon → train predictions
cb_base_preds_vl = {}
cb_base_preds_ts = {}
cb_base_models   = {}

for h_label in H_LABELS:
    y_tr_h = mh_train[["dLAT_3h","dLON_3h"]].values if h_label == "3h" \
             else mh_train[[f"dLAT_{h_label}",f"dLON_{h_label}"]].values
    y_vl_h = mh_val[["dLAT_3h","dLON_3h"]].values if h_label == "3h" \
             else mh_val[[f"dLAT_{h_label}",f"dLON_{h_label}"]].values

    cb = CatBoostRegressor(**CB_P)
    cb.fit(Xmh_tr, y_tr_h, eval_set=(Xmh_vl, y_vl_h))
    cb_base_preds_tr[h_label] = cb.predict(Xmh_tr)
    cb_base_preds_vl[h_label] = cb.predict(Xmh_vl)
    cb_base_preds_ts[h_label] = cb.predict(Xmh_ts)
    cb_base_models[h_label]   = cb
    print(f"    CatBoost base {h_label} trained")

# --- Step 2 & 3: Build correction features and train NN per horizon ---
# Motion feature indices in ALL_FEATURE_COLS for augmenting correction input
motion_feat_names = ["geo_speed_kmh","geo_bear_sin","geo_bear_cos",
                     "curvature","dlat_accel","dlon_accel",
                     "dLAT_lag1","dLON_lag1","dist_BoB"]
motion_feat_idx = [ALL_FEATURE_COLS.index(c) for c in motion_feat_names
                   if c in ALL_FEATURE_COLS]
print(f"    Motion features for correction: {len(motion_feat_idx)}")

def build_correction_features(X_scaled, cb_pred_deg):
    """
    Concatenate:
      - CatBoost predicted dLAT/dLON converted to km
      - Storm motion features from scaled input
    Returns correction input matrix.
    """
    cb_km = cb_pred_deg * np.array([DEG_TO_KM_LAT, DEG_TO_KM_LON])
    motion = X_scaled[:, motion_feat_idx] if len(motion_feat_idx) > 0 \
             else np.zeros((len(X_scaled), 1))
    return np.hstack([cb_km, motion]).astype(np.float32)

def build_correction_nn(n_corr_feat):
    """
    Small, stable correction NN.
    Input: [CB_pred_lat_km, CB_pred_lon_km, motion_features]
    Output: [delta_lat_km, delta_lon_km]  (correction to add to CB)
    LR=0.01 with clipnorm is fine here — correction target is small
    and well-scaled in km.
    """
    inp = Input((n_corr_feat,))
    x   = Dense(64, activation=ACT)(inp)
    x   = BatchNormalization()(x)
    x   = Dropout(D)(x)
    x   = Dense(32, activation=ACT)(x)
    x   = Dropout(D)(x)
    x   = Dense(16, activation=ACT)(x)
    out = Dense(2)(x)
    m   = Model(inp, out)
    m.compile(AdamW(learning_rate=GLOBAL_LR, clipnorm=1.0), loss="mse")
    return m

cbnn_models = {}

for h_label in H_LABELS:
    y_tr_h = mh_train[["dLAT_3h","dLON_3h"]].values if h_label == "3h" \
             else mh_train[[f"dLAT_{h_label}",f"dLON_{h_label}"]].values
    y_vl_h = mh_val[["dLAT_3h","dLON_3h"]].values if h_label == "3h" \
             else mh_val[[f"dLAT_{h_label}",f"dLON_{h_label}"]].values
    y_ts_h = mh_test[["dLAT_3h","dLON_3h"]].values if h_label == "3h" \
             else mh_test[[f"dLAT_{h_label}",f"dLON_{h_label}"]].values

    lc = latmh_ts; ln = lonmh_ts

    # Build correction features
    Xc_tr = build_correction_features(Xmh_tr, cb_base_preds_tr[h_label])
    Xc_vl = build_correction_features(Xmh_vl, cb_base_preds_vl[h_label])
    Xc_ts = build_correction_features(Xmh_ts, cb_base_preds_ts[h_label])

    # Residual targets in km space
    res_tr_km = (y_tr_h - cb_base_preds_tr[h_label]) * \
                np.array([DEG_TO_KM_LAT, DEG_TO_KM_LON])
    res_vl_km = (y_vl_h - cb_base_preds_vl[h_label]) * \
                np.array([DEG_TO_KM_LAT, DEG_TO_KM_LON])

    # Train correction NN
    nn = build_correction_nn(Xc_tr.shape[1])
    nn.fit(
        Xc_tr.astype(np.float32),
        res_tr_km.astype(np.float32),
        validation_data=(Xc_vl.astype(np.float32),
                         res_vl_km.astype(np.float32)),
        epochs=GLOBAL_EPOCHS,
        batch_size=GLOBAL_BATCH,
        verbose=0,
        callbacks=[]    # EarlyStopping OFF — fairness
    )
    cbnn_models[h_label] = nn

    # Final prediction = CB base + NN correction (converted back to degrees)
    corr_km  = nn.predict(Xc_ts.astype(np.float32), verbose=0)
    corr_deg = corr_km / np.array([DEG_TO_KM_LAT, DEG_TO_KM_LON])
    cbnn_pred = cb_base_preds_ts[h_label] + corr_deg

    rec_ml_horizon("CB+MotionNN", h_label, lc, ln, y_ts_h, cbnn_pred)
    m = all_results[h_label]["CB+MotionNN"]
    print(f"    {h_label}: Med={m['median_km']:.2f}  "
          f"R2lat={m['r2_dlat']:.3f}  R2lon={m['r2_dlon']:.3f}")

# ── 8B. SECE Ensemble — Full optimization ───────────────────
# Improvements:
#   1. RF added as 4th base learner (RF was rank #1-2 every run)
#   2. "trend" feature subset using new long-horizon features
#   3. Stacking Ensemble val/test preds as extra base learner
#   4. Horizon-specific Ridge alpha:
#        3h=1.0 (tight regularization, short-range is clean)
#        12h=0.5, 24h=0.1, 48h=0.01 (looser = aggressive best-picker)
print("\n  ▶ SECE Ensemble (CB+XGB+LGB+RF+BLSTM+CNN-GRU bases, 6 subsets, Stacking+PRC extra)")

F = ALL_FEATURE_COLS

# Horizon-specific Ridge alpha
# Ridge alpha only needed for 24h/48h (defined in hybrid meta block below)

sece_subsets = {
    "position" : [c for c in F if any(k in c for k in
                  ["LAT","LON","dist_BoB","lat_lon"])],
    "motion"   : [c for c in F if any(k in c for k in
                  ["dLAT","dLON","geo_","speed","bearing","accel","curvature"])],
    "physics"  : [c for c in F if any(k in c for k in
                  ["curvature","bearing_stab","speed_stab","season","bearing_con"])],
    "environ"  : [c for c in F if any(k in c for k in
                  ["DIST2LAND","LANDFALL","NEWDELHI","STORM_SPEED","STORM_DIR"])],
    "trend"    : [c for c in F if any(k in c for k in
                  ["trend","momentum","recurvature","dist2land_rate","accel"])],
    "full"     : F,
}
sece_subsets = {k: v for k, v in sece_subsets.items() if len(v) >= 2}
print(f"    Feature subsets: {[(k, len(v)) for k, v in sece_subsets.items()]}")

RF_P = dict(n_estimators=300, n_jobs=-1, random_state=42)

def make_sece_bases():
    return [
        ("CB",  CatBoostRegressor(**CB_P)),
        ("XGB", MultiOutputRegressor(xgb.XGBRegressor(**XGB_P))),
        ("LGB", MultiOutputRegressor(lgb.LGBMRegressor(**LGB_P))),
        ("RF",  MultiOutputRegressor(RandomForestRegressor(**RF_P))),
    ]

sece_val_preds_by_h  = {h: [] for h in H_LABELS}
sece_test_preds_by_h = {h: [] for h in H_LABELS}
sece_base_preds      = {}

# --- Compute PRC predictions (extra base learner) ---
prc_val_preds  = {}
prc_test_preds = {}

for h_label in H_LABELS:
    y_tr_h = mh_train[["dLAT_3h","dLON_3h"]].values if h_label == "3h" \
             else mh_train[[f"dLAT_{h_label}",f"dLON_{h_label}"]].values
    y_vl_h = mh_val[["dLAT_3h","dLON_3h"]].values if h_label == "3h" \
             else mh_val[[f"dLAT_{h_label}",f"dLON_{h_label}"]].values

    persist_tr   = np.column_stack([mh_train["dLAT_lag1"].fillna(0).values,
                                     mh_train["dLON_lag1"].fillna(0).values])
    persist_vl   = np.column_stack([mh_val["dLAT_lag1"].fillna(0).values,
                                     mh_val["dLON_lag1"].fillna(0).values])
    persist_ts_h = np.column_stack([mh_test["dLAT_lag1"].fillna(0).values,
                                     mh_test["dLON_lag1"].fillna(0).values])

    res_tr_km = (y_tr_h - persist_tr) * np.array([DEG_TO_KM_LAT, DEG_TO_KM_LON])

    # Use XGBoost for 48h residual (rank #2 at 48h), LGB for others
    if h_label == "48h":
        pl = xgb.XGBRegressor(**XGB_P); pn = xgb.XGBRegressor(**XGB_P)
    else:
        pl = lgb.LGBMRegressor(**LGB_P); pn = lgb.LGBMRegressor(**LGB_P)
    pl.fit(Xmh_tr, res_tr_km[:, 0]); pn.fit(Xmh_tr, res_tr_km[:, 1])

    res_pred_vl = np.column_stack([pl.predict(Xmh_vl),
                                    pn.predict(Xmh_vl)]) / \
                  np.array([DEG_TO_KM_LAT, DEG_TO_KM_LON])
    prc_val_preds[h_label] = persist_vl + res_pred_vl

    res_pred_ts = np.column_stack([pl.predict(Xmh_ts),
                                    pn.predict(Xmh_ts)]) / \
                  np.array([DEG_TO_KM_LAT, DEG_TO_KM_LON])
    prc_test_preds[h_label] = persist_ts_h + res_pred_ts

# --- Train SECE base learners ---
for h_label in H_LABELS:
    y_tr_h = mh_train[["dLAT_3h","dLON_3h"]].values if h_label == "3h" \
             else mh_train[[f"dLAT_{h_label}",f"dLON_{h_label}"]].values
    y_vl_h = mh_val[["dLAT_3h","dLON_3h"]].values if h_label == "3h" \
             else mh_val[[f"dLAT_{h_label}",f"dLON_{h_label}"]].values

    base_preds_h = []
    for sname, cols in sece_subsets.items():
        idx = [F.index(c) for c in cols if c in F]
        if len(idx) < 2: continue
        Xtr_s = Xmh_tr[:, idx]; Xvl_s = Xmh_vl[:, idx]; Xts_s = Xmh_ts[:, idx]
        for alg_name, alg in make_sece_bases():
            if alg_name == "CB":
                alg.fit(Xtr_s, y_tr_h, eval_set=(Xvl_s, y_vl_h))
            else:
                alg.fit(Xtr_s, y_tr_h)
            sece_val_preds_by_h[h_label].append(alg.predict(Xvl_s))
            tp = alg.predict(Xts_s)
            sece_test_preds_by_h[h_label].append(tp)
            base_preds_h.append(tp)

    # Extra base learners: PRC + Stacking Ensemble
    sece_val_preds_by_h[h_label].append(prc_val_preds[h_label])
    sece_test_preds_by_h[h_label].append(prc_test_preds[h_label])
    base_preds_h.append(prc_test_preds[h_label])

    if h_label in stk_val_preds:
        sece_val_preds_by_h[h_label].append(stk_val_preds[h_label])
        sece_test_preds_by_h[h_label].append(stk_test_preds[h_label])
        base_preds_h.append(stk_test_preds[h_label])

    # ── DL base learners (BLSTM + CNN-GRU) ──
    # DL models run on seq-aligned data (n_seq < n_mh) so we need to
    # pad to mh size: fill the first SEQ_LEN-1 rows with the mean pred
    # so shape matches mh_val/mh_test for the Ridge/LGB meta-learner.
    for dl_name in SECE_DL_MODELS:
        if dl_name in dl_sece_val_preds and h_label in dl_sece_val_preds[dl_name]:
            n_mh_vl = len(mh_val)
            n_mh_ts = len(mh_test)

            vl_seq  = dl_sece_val_preds[dl_name][h_label]   # (n_seq_vl, 2)
            ts_seq  = dl_sece_test_preds[dl_name][h_label]  # (n_seq_ts, 2)

            pad_vl  = n_mh_vl - len(vl_seq)
            pad_ts  = n_mh_ts - len(ts_seq)

            # Pad front with column mean (conservative neutral prediction)
            vl_full = np.vstack([np.tile(vl_seq.mean(axis=0), (pad_vl, 1)), vl_seq])
            ts_full = np.vstack([np.tile(ts_seq.mean(axis=0), (pad_ts, 1)), ts_seq])

            sece_val_preds_by_h[h_label].append(vl_full)
            sece_test_preds_by_h[h_label].append(ts_full)
            base_preds_h.append(ts_full)

    sece_base_preds[h_label] = base_preds_h

# --- Hybrid meta-learner: LGB for 3h/12h, Ridge for 24h/48h ──
# LightGBM meta wins at short horizons (3h/12h) — large val set, clean signal
# Ridge meta wins at long horizons (24h/48h) — LGB overfits on small val patterns
# Context features appended for LGB meta only
# No leakage: all meta-learners trained on VAL predictions only

CTX_FEAT = ["LAT","LON","DIST2LAND","geo_speed_kmh","season_sin","season_cos",
            "dlat_trend","dlon_trend","recurvature_proxy"]
ctx_idx  = [ALL_FEATURE_COLS.index(c) for c in CTX_FEAT if c in ALL_FEATURE_COLS]
ctx_vl   = Xmh_vl[:, ctx_idx]
ctx_ts   = Xmh_ts[:, ctx_idx]
print(f"    Meta context features ({len(ctx_idx)}): {[ALL_FEATURE_COLS[i] for i in ctx_idx]}")

LGB_META = dict(n_estimators=300, learning_rate=GLOBAL_LR,
                max_depth=4, num_leaves=15,
                min_child_samples=20,
                subsample=0.8, colsample_bytree=0.8,
                random_state=42, verbose=-1)

RIDGE_ALPHA = {"24h": 0.1, "48h": 0.01}

for h_label in H_LABELS:
    y_vl_h = mh_val[["dLAT_3h","dLON_3h"]].values if h_label == "3h" \
             else mh_val[[f"dLAT_{h_label}",f"dLON_{h_label}"]].values
    y_ts_h = mh_test[["dLAT_3h","dLON_3h"]].values if h_label == "3h" \
             else mh_test[[f"dLAT_{h_label}",f"dLON_{h_label}"]].values

    Xm_vl_base = np.hstack(sece_val_preds_by_h[h_label])
    Xm_ts_base = np.hstack(sece_test_preds_by_h[h_label])

    if h_label in ["3h", "12h"]:
        # LightGBM meta + storm context — best at short horizons
        Xm_vl = np.hstack([Xm_vl_base, ctx_vl])
        Xm_ts = np.hstack([Xm_ts_base, ctx_ts])
        meta_lat = lgb.LGBMRegressor(**LGB_META)
        meta_lon = lgb.LGBMRegressor(**LGB_META)
        meta_lat.fit(Xm_vl, y_vl_h[:, 0])
        meta_lon.fit(Xm_vl, y_vl_h[:, 1])
        sece_pred = np.column_stack([meta_lat.predict(Xm_ts),
                                      meta_lon.predict(Xm_ts)])
        meta_type = f"LGB+ctx"
    else:
        # Ridge meta — better at long horizons (avoids overfitting small val)
        alpha = RIDGE_ALPHA[h_label]
        ml_lat = Ridge(alpha=alpha); ml_lat.fit(Xm_vl_base, y_vl_h[:, 0])
        ml_lon = Ridge(alpha=alpha); ml_lon.fit(Xm_vl_base, y_vl_h[:, 1])
        sece_pred = np.column_stack([ml_lat.predict(Xm_ts_base),
                                      ml_lon.predict(Xm_ts_base)])
        meta_type = f"Ridge(a={alpha})"

    rec_ml_horizon("SECE Ensemble", h_label, latmh_ts, lonmh_ts, y_ts_h, sece_pred)
    m = all_results[h_label]["SECE Ensemble"]
    print(f"    {h_label}: Med={m['median_km']:.2f}  "
          f"R2lat={m['r2_dlat']:.3f}  meta={meta_type}")

# ── 8C. Persistence Residual Cascade (PRC) ──
# km-scaled residual — XGBoost for 48h, LightGBM for 3h/12h/24h
print("\n  ▶ Persistence Residual Cascade (XGB@48h, LGB@others, km-scaled)")

for h_label in H_LABELS:
    if h_label == "3h":
        persist_tr   = np.column_stack([df_tr3["dLAT_lag1"].fillna(0).values,
                                         df_tr3["dLON_lag1"].fillna(0).values])
        persist_ts_h = persist_3h
        y_tr_h = y3_tr; y_ts_h = y3_ts
        Xtr_h = X3_tr; Xts_h = X3_ts
        lc = lat3_ts; ln = lon3_ts
    else:
        persist_tr   = np.column_stack([mh_train["dLAT_lag1"].fillna(0).values,
                                         mh_train["dLON_lag1"].fillna(0).values])
        persist_ts_h = np.column_stack([mh_test["dLAT_lag1"].fillna(0).values,
                                         mh_test["dLON_lag1"].fillna(0).values])
        y_tr_h = mh_train[[f"dLAT_{h_label}",f"dLON_{h_label}"]].values
        y_ts_h = mh_test[[f"dLAT_{h_label}",f"dLON_{h_label}"]].values
        Xtr_h = Xmh_tr; Xts_h = Xmh_ts
        lc = latmh_ts; ln = lonmh_ts

    res_tr_km = (y_tr_h - persist_tr) * np.array([DEG_TO_KM_LAT, DEG_TO_KM_LON])

    # XGBoost for 48h (rank #2 at 48h), LightGBM for others
    if h_label == "48h":
        prc_lat = xgb.XGBRegressor(**XGB_P)
        prc_lon = xgb.XGBRegressor(**XGB_P)
    else:
        prc_lat = lgb.LGBMRegressor(**LGB_P)
        prc_lon = lgb.LGBMRegressor(**LGB_P)

    prc_lat.fit(Xtr_h, res_tr_km[:, 0])
    prc_lon.fit(Xtr_h, res_tr_km[:, 1])

    pred_res_deg = np.column_stack([
        prc_lat.predict(Xts_h),
        prc_lon.predict(Xts_h)
    ]) / np.array([DEG_TO_KM_LAT, DEG_TO_KM_LON])

    prc_pred = persist_ts_h + pred_res_deg
    rec_ml_horizon("Persistence Residual Cascade", h_label, lc, ln, y_ts_h, prc_pred)
    m = all_results[h_label]["Persistence Residual Cascade"]
    print(f"    {h_label}: Med={m['median_km']:.2f}  R2lat={m['r2_dlat']:.3f}")


8. CUSTOM MODELS
  Variables found from Sections 1-7 — skipping rebuild

  ▶ CB+MotionNN (CatBoost + per-horizon bias correction NN)
    CatBoost base 3h trained
    CatBoost base 12h trained
    CatBoost base 24h trained
    CatBoost base 48h trained
    Motion features for correction: 9


E0000 00:00:1772689289.760475    3145 buffer_comparator.cc:156] Difference at 0: 1.23795, expected 0.835012
E0000 00:00:1772689289.760519    3145 buffer_comparator.cc:156] Difference at 1: 0.641268, expected 0.442422
E0000 00:00:1772689289.760523    3145 buffer_comparator.cc:156] Difference at 2: 1.42676, expected 0.942373
E0000 00:00:1772689289.760525    3145 buffer_comparator.cc:156] Difference at 3: 0.74123, expected 0.502213
E0000 00:00:1772689289.760527    3145 buffer_comparator.cc:156] Difference at 6: 1.35822, expected 0.898864
E0000 00:00:1772689289.760529    3145 buffer_comparator.cc:156] Difference at 7: 0.704142, expected 0.477513
E0000 00:00:1772689289.760531    3145 buffer_comparator.cc:156] Difference at 8: 1.32059, expected 0.868683
E0000 00:00:1772689289.760533    3145 buffer_comparator.cc:156] Difference at 9: 0.697293, expected 0.474336
E0000 00:00:1772689289.760535    3145 buffer_comparator.cc:156] Difference at 10: 0.834453, expected 0.60827
E0000 00:00:1772689289.7

    3h: Med=5.91  R2lat=0.881  R2lon=0.953
    12h: Med=31.11  R2lat=0.753  R2lon=0.835
    24h: Med=95.75  R2lat=0.557  R2lon=0.640
    48h: Med=234.06  R2lat=0.379  R2lon=0.534

  ▶ SECE Ensemble (CB+XGB+LGB+RF+BLSTM+CNN-GRU bases, 6 subsets, Stacking+PRC extra)
    Feature subsets: [('position', 24), ('motion', 24), ('physics', 7), ('environ', 6), ('trend', 8), ('full', 49)]
    Meta context features (9): ['LAT', 'LON', 'DIST2LAND', 'geo_speed_kmh', 'season_sin', 'season_cos', 'dlat_trend', 'dlon_trend', 'recurvature_proxy']
    3h: Med=4.40  R2lat=0.911  meta=LGB+ctx
    12h: Med=30.00  R2lat=0.774  meta=LGB+ctx
    24h: Med=86.44  R2lat=0.576  meta=Ridge(a=0.1)
    48h: Med=228.88  R2lat=0.343  meta=Ridge(a=0.01)

  ▶ Persistence Residual Cascade (XGB@48h, LGB@others, km-scaled)
    3h: Med=7.77  R2lat=0.861
    12h: Med=30.90  R2lat=0.754
    24h: Med=88.70  R2lat=0.571
    48h: Med=227.64  R2lat=0.407


In [5]:
# 9. RANK CHECK — CONSISTENT WINNER PROOF
# ─────────────────────────────────────────────────────────────
print("\n"+"="*60)
print("9. CONSISTENT WINNER ANALYSIS")
print("="*60)
custom_models = ["CB+MotionNN", "SECE Ensemble", "Persistence Residual Cascade"]
all_model_names = list(set(
    list(all_results["3h"].keys()) +
    list(all_results["12h"].keys())
))

for cm in custom_models:
    print(f"\n  {cm}:")
    wins = 0
    for h_label in H_LABELS:
        if h_label not in all_results or cm not in all_results[h_label]: continue
        sorted_models = sorted(all_results[h_label].items(),
                               key=lambda x: x[1]["median_km"])
        rank = [m[0] for m in sorted_models].index(cm) + 1
        med  = all_results[h_label][cm]["median_km"]
        best = sorted_models[0][1]["median_km"]
        diff = med - best
        print(f"    {h_label}: rank #{rank}  Med={med:.2f}  Best={best:.2f}  diff={diff:+.2f}")
        if rank == 1: wins += 1
    print(f"    Wins: {wins}/{len(H_LABELS)}")



9. CONSISTENT WINNER ANALYSIS

  CB+MotionNN:
    3h: rank #5  Med=5.91  Best=4.40  diff=+1.51
    12h: rank #6  Med=31.11  Best=29.88  diff=+1.23
    24h: rank #9  Med=95.75  Best=86.44  diff=+9.30
    48h: rank #7  Med=234.06  Best=227.19  diff=+6.88
    Wins: 0/4

  SECE Ensemble:
    3h: rank #1  Med=4.40  Best=4.40  diff=+0.00
    12h: rank #2  Med=30.00  Best=29.88  diff=+0.13
    24h: rank #1  Med=86.44  Best=86.44  diff=+0.00
    48h: rank #4  Med=228.88  Best=227.19  diff=+1.69
    Wins: 2/4

  Persistence Residual Cascade:
    3h: rank #9  Med=7.77  Best=4.40  diff=+3.37
    12h: rank #5  Med=30.90  Best=29.88  diff=+1.03
    24h: rank #5  Med=88.70  Best=86.44  diff=+2.26
    48h: rank #2  Med=227.64  Best=227.19  diff=+0.45
    Wins: 0/4


In [6]:
# ─────────────────────────────────────────────────────────────
# 10. SAVE METRICS
# ─────────────────────────────────────────────────────────────
print("\n[10] Saving metrics …")
rows_3h=[]
for name,m in all_results["3h"].items():
    rows_3h.append({"model":name,"median_km":m["median_km"],"mean_km":m["mean_km"],
                    "p75_km":m["p75_km"],"p90_km":m["p90_km"],
                    "rmse_dlat":m["rmse_dlat"],"rmse_dlon":m["rmse_dlon"],
                    "r2_dlat":m["r2_dlat"],"r2_dlon":m["r2_dlon"]})
pd.DataFrame(rows_3h).sort_values("median_km").to_csv(
    "outputs/metrics/metrics_3h_standardized.csv", index=False)

rows_ro=[]
for h in H_LABELS[1:]:
    for name,m in all_results[h].items():
        rows_ro.append({"horizon":h,"model":name,
                        "median_km":m["median_km"],"mean_km":m["mean_km"],
                        "p75_km":m["p75_km"],"p90_km":m["p90_km"],
                        "rmse_dlat":m["rmse_dlat"],"rmse_dlon":m["rmse_dlon"],
                        "r2_dlat":m["r2_dlat"],"r2_dlon":m["r2_dlon"]})
pd.DataFrame(rows_ro).to_csv("outputs/metrics/metrics_rollout_direct_standardized.csv", index=False)
print("  ✓ metrics_3h_standardized.csv")
print("  ✓ metrics_rollout_direct_standardized.csv")

# ─────────────────────────────────────────────────────────────
# 11. SAVE PREDICTIONS
# ─────────────────────────────────────────────────────────────
print("\n[11] Saving predictions …")

# 3h
base3 = df_ts3[["SID","ISO_TIME","LAT","LON","LAT_next","LON_next",
                 "dLAT","dLON"]].copy().reset_index(drop=True)
for name, p in all_preds["3h"].items():
    if len(p) == len(base3):
        base3[f"pred_dLAT_{name}"] = p[:,0]; base3[f"pred_dLON_{name}"] = p[:,1]
    else:
        cl=np.full(len(base3),np.nan); cn=np.full(len(base3),np.nan)
        cl[SEQ_LEN-1:SEQ_LEN-1+len(p)]=p[:,0]; cn[SEQ_LEN-1:SEQ_LEN-1+len(p)]=p[:,1]
        base3[f"pred_dLAT_{name}"]=cl; base3[f"pred_dLON_{name}"]=cn
base3.to_csv("outputs/predictions/predictions_3h_all_models.csv", index=False)

# Rollout horizons
for h_label in H_LABELS[1:]:
    baseh = mh_test[["SID","ISO_TIME","LAT","LON",
                      f"LAT_{h_label}",f"LON_{h_label}",
                      f"dLAT_{h_label}",f"dLON_{h_label}"]].copy().reset_index(drop=True)
    for name, p in all_preds[h_label].items():
        if len(p)==len(baseh):
            baseh[f"pred_dLAT_{name}"]=p[:,0]; baseh[f"pred_dLON_{name}"]=p[:,1]
        else:
            cl=np.full(len(baseh),np.nan); cn=np.full(len(baseh),np.nan)
            cl[SEQ_LEN-1:SEQ_LEN-1+len(p)]=p[:,0]; cn[SEQ_LEN-1:SEQ_LEN-1+len(p)]=p[:,1]
            baseh[f"pred_dLAT_{name}"]=cl; baseh[f"pred_dLON_{name}"]=cn
    baseh.to_csv(f"outputs/predictions/predictions_{h_label}_all_models.csv", index=False)
    print(f"  ✓ predictions_{h_label}_all_models.csv")

print("  ✓ predictions_3h_all_models.csv")



[10] Saving metrics …
  ✓ metrics_3h_standardized.csv
  ✓ metrics_rollout_direct_standardized.csv

[11] Saving predictions …
  ✓ predictions_12h_all_models.csv
  ✓ predictions_24h_all_models.csv
  ✓ predictions_48h_all_models.csv
  ✓ predictions_3h_all_models.csv


In [ ]:
# ─────────────────────────────────────────────────────────────
# 12. CATROPY COMPUTATION
# ─────────────────────────────────────────────────────────────
print("\n"+"="*60)
print("12. CATROPY — TRAJECTORY ENTROPY / STABILITY")
print("="*60)
"""
Catropy definition (defensible, deterministic):
For each model and horizon h:
  For each test storm:
    1. Bearing entropy: compute predicted bearings from consecutive
       predicted positions, discretise to 8 bins, compute Shannon entropy.
    2. Curvature entropy: discretise bearing changes to 8 bins, entropy.
  Catropy = mean bearing entropy + mean curvature entropy across storms.

For SECE: also compute spatial spread entropy from base learner endpoints.
Higher catropy = more uncertain/variable predictions (more spread).
"""

def bearing_from_dlat_dlon(dlat_arr, dlon_arr):
    return np.degrees(np.arctan2(dlon_arr, dlat_arr)) % 360

def compute_catropy_for_preds(sid_arr, lat_arr, preds_dlat, preds_dlon):
    """
    sid_arr, lat_arr are row-wise; preds are arrays of same length.
    Returns mean catropy (bearing entropy + curvature entropy).
    """
    storms = np.unique(sid_arr)
    b_entropies=[]; c_entropies=[]
    for sid in storms:
        mask = sid_arr == sid
        dl = preds_dlat[mask]; dn = preds_dlon[mask]
        if len(dl) < 3: continue
        bearings = bearing_from_dlat_dlon(dl, dn)
        # Bearing entropy
        counts, _ = np.histogram(bearings, bins=16, range=(0, 360))
        counts = counts + 1e-9   # smoothing
        b_ent  = scipy_entropy(counts / counts.sum())
        b_entropies.append(b_ent)
        # Curvature entropy (bearing changes)
        bear_changes = np.diff(bearings)
        bear_changes = (bear_changes + 180) % 360 - 180  # wrap to (-180, 180)
        counts2, _ = np.histogram(bear_changes, bins=16, range=(-180, 180))
        counts2 = counts2 + 1e-9
        c_ent = scipy_entropy(counts2 / counts2.sum())
        c_entropies.append(c_ent)
    return float(np.mean(b_entropies)), float(np.mean(c_entropies))

catropy_rows = []
COLORS = plt.cm.tab20.colors

for h_label in H_LABELS:
    if h_label == "3h":
        sid_arr = df_ts3["SID"].values
        lc_arr  = lat3_ts
    else:
        sid_arr = mh_test["SID"].values
        lc_arr  = latmh_ts

    for name, m in all_results[h_label].items():
        p = all_preds[h_label].get(name)
        if p is None or len(p) < 10: continue
        # Align lengths
        n_rows = min(len(sid_arr), len(p))
        be, ce = compute_catropy_for_preds(
            sid_arr[:n_rows], lc_arr[:n_rows], p[:n_rows,0], p[:n_rows,1])
        catropy_total = be + ce
        catropy_rows.append({
            "horizon": h_label, "model": name,
            "bearing_entropy": round(be, 5),
            "curvature_entropy": round(ce, 5),
            "catropy": round(catropy_total, 5),
            "median_km": m["median_km"]
        })

# SECE spatial spread entropy
for h_label in H_LABELS:
    if h_label not in sece_base_preds: continue
    base_ps = sece_base_preds[h_label]
    if len(base_ps) < 2: continue
    # Endpoint spread: std of predicted final lat/lon across base learners
    endpoints_lat = np.stack([p[:,0] for p in base_ps], axis=1)
    endpoints_lon = np.stack([p[:,1] for p in base_ps], axis=1)
    spread_lat = np.std(endpoints_lat, axis=1)
    spread_lon = np.std(endpoints_lon, axis=1)
    counts_lat, _ = np.histogram(spread_lat, bins=16)
    counts_lat = counts_lat + 1e-9
    spread_ent = float(scipy_entropy(counts_lat / counts_lat.sum()))
    # Update SECE row
    for row in catropy_rows:
        if row["horizon"] == h_label and row["model"] == "SECE Ensemble":
            row["catropy"]           = round(row["catropy"] + spread_ent, 5)
            row["spread_entropy"]    = round(spread_ent, 5)

catropy_df = pd.DataFrame(catropy_rows)
catropy_df.to_csv("outputs/predictions/catropy_scores_all_models.csv", index=False)
print(f"  ✓ catropy_scores_all_models.csv  ({len(catropy_df)} rows)")

# ─────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────
# 13. PLOTS
# ─────────────────────────────────────────────────────────────
print("\n"+"="*60)
print("13. GENERATING ALL PLOTS")
print("="*60)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
os.makedirs("outputs/plots", exist_ok=True)

# ── Safe defaults if running standalone ──
if "dl_history"     not in dir(): dl_history   = {}
if "cbnn_models"    not in dir(): cbnn_models  = {}
if "SEQ_LEN"        not in dir(): SEQ_LEN      = 4
if "GLOBAL_EPOCHS"  not in dir(): GLOBAL_EPOCHS= 300
if "H_LABELS"       not in dir(): H_LABELS     = ["3h","12h","24h","48h"]
if "H_HOURS"        not in dir(): H_HOURS      = [3,12,24,48]
if "custom_models"  not in dir():
    custom_models = ["CB+MotionNN","SECE Ensemble",
                     "Persistence Residual Cascade","SECE+PRC Meta"]

COLORS       = plt.cm.tab20.colors
TRACK_COLORS = plt.cm.Set1.colors

# Rebuild rows_3h from all_results
rows_3h = [{"model":n,"median_km":m["median_km"],"mean_km":m["mean_km"],
             "p75_km":m["p75_km"],"p90_km":m["p90_km"],
             "rmse_dlat":m["rmse_dlat"],"rmse_dlon":m["rmse_dlon"],
             "r2_dlat":m["r2_dlat"],"r2_dlon":m["r2_dlon"]}
            for n,m in all_results["3h"].items()]

# Model category lists — built from whatever is in all_results
_dl_set = {"LSTM","GRU","BLSTM","SLSTM","CNN","CNN-GRU","CNN-LSTM",
           "GRU-LSTM","Hybrid Transformer"}
dl_names = [k for k in all_results["3h"] if k in _dl_set]
ml_names = [k for k in all_results["3h"] if k not in _dl_set and k not in custom_models]

def sort_by_3h(names):
    valid = [n for n in names if n in all_results["3h"]]
    return sorted(valid, key=lambda n: all_results["3h"][n]["median_km"])

top4_ml = sort_by_3h([n for n in ml_names
                       if "Persistence" not in n and "Stacking" not in n])[:4]
top4_dl = sort_by_3h(dl_names)[:4]

# ── P1: Model comparison bar — 3h median ──
rows_3h_sorted = pd.DataFrame(rows_3h).sort_values("median_km")
fig, ax = plt.subplots(figsize=(14, max(6, len(rows_3h_sorted)*0.45)))
bars = ax.barh(rows_3h_sorted["model"], rows_3h_sorted["median_km"],
               color=[COLORS[i%20] for i in range(len(rows_3h_sorted))],
               edgecolor="black", linewidth=0.5)
ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)
ax.set_xlabel("Median Haversine Error (km)", fontsize=12)
ax.set_title("3h Prediction — All Models\n"
             "[Standardized: AdamW|LR=0.001|ReLU|Drop=0.2|Ep=300|NoES]",
             fontsize=11, fontweight="bold")
ax.invert_yaxis(); ax.grid(axis="x", alpha=0.4)
for bar, label in zip(ax.patches, rows_3h_sorted["model"].tolist()):
    if label in custom_models:
        bar.set_edgecolor("red"); bar.set_linewidth(2.5)
ax.legend(handles=[
    plt.Rectangle((0,0),1,1,fc="white",ec="red",lw=2.5,label="Custom models")
], fontsize=9)
plt.tight_layout()
plt.savefig("outputs/plots/p1_model_comparison_3h.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ p1_model_comparison_3h.png")

# ── P2: Error vs horizon ──
# Build groups dynamically based on what models are available
plot_groups = []
if top4_ml: plot_groups.append(("Top ML Baselines", top4_ml))
if top4_dl: plot_groups.append(("Top DL Baselines", top4_dl))
plot_groups.append(("Custom Models", custom_models))

n_groups = len(plot_groups)
fig, axes = plt.subplots(1, n_groups, figsize=(7*n_groups, 7))
if n_groups == 1: axes = [axes]
fig.suptitle("Error vs Lead Time — Best Models per Category",
             fontsize=14, fontweight="bold")
for ax_i, (group_name, group_models) in enumerate(plot_groups):
    ax = axes[ax_i]
    for mi, mname in enumerate(group_models):
        xv=[]; yv=[]
        for h_label, h_hr in zip(H_LABELS, H_HOURS):
            if mname in all_results.get(h_label,{}):
                xv.append(h_hr)
                yv.append(all_results[h_label][mname]["median_km"])
        if yv:
            lw = 2.5 if mname in custom_models else 1.8
            ax.plot(xv, yv, marker="o", lw=lw, label=mname, color=COLORS[mi%20])
    ax.set_xlabel("Lead Time (hours)"); ax.set_ylabel("Median Error (km)")
    ax.set_title(group_name, fontsize=12, fontweight="bold")
    ax.set_xticks(H_HOURS); ax.legend(fontsize=8); ax.grid(alpha=0.4)
plt.tight_layout()
plt.savefig("outputs/plots/p2_error_vs_horizon.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ p2_error_vs_horizon.png")

# ── P3: Catropy vs horizon ──
if "catropy_df" in dir() and len(catropy_df) > 0:
    fig, ax = plt.subplots(figsize=(14, 7))
    for mi, mname in enumerate(catropy_df["model"].unique()):
        sub = catropy_df[catropy_df["model"]==mname].sort_values(
              "horizon", key=lambda x: x.map({"3h":0,"12h":1,"24h":2,"48h":3}))
        lw = 2.5 if mname in custom_models else 1.2
        ls = "-"  if mname in custom_models else "--"
        ax.plot(sub["horizon"], sub["catropy"], marker="o", lw=lw, ls=ls,
                label=mname, color=COLORS[mi%20])
    ax.set_xlabel("Horizon"); ax.set_ylabel("Catropy (bearing + curvature entropy)")
    ax.set_title("Catropy vs Horizon — All Models\n"
                 "(Higher = more uncertain/variable predictions)",
                 fontsize=13, fontweight="bold")
    ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.4)
    plt.tight_layout()
    plt.savefig("outputs/plots/p3_catropy_vs_horizon.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("  ✓ p3_catropy_vs_horizon.png")
else:
    print("  ⚠ p3 skipped — catropy_df not available (run Section 12 first)")

# ── P4: Catropy vs error scatter ──
if "catropy_df" in dir() and len(catropy_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    fig.suptitle("Catropy vs Median Error — Stability Analysis",
                 fontsize=14, fontweight="bold")
    for ax_i, h_label in enumerate(["3h","48h"]):
        ax = axes[ax_i]
        sub = catropy_df[catropy_df["horizon"]==h_label]
        for _, row in sub.iterrows():
            color = "red"  if row["model"] in custom_models else \
                    "blue" if row["model"] in _dl_set else "gray"
            ax.scatter(row["catropy"], row["median_km"], color=color, s=80, zorder=4)
            ax.annotate(row["model"], (row["catropy"], row["median_km"]),
                        fontsize=6, xytext=(4,2), textcoords="offset points")
        ax.set_xlabel("Catropy"); ax.set_ylabel("Median Error (km)")
        ax.set_title(h_label, fontsize=12, fontweight="bold"); ax.grid(alpha=0.4)
        ax.scatter([],[],color="red",  s=60, label="Custom")
        ax.scatter([],[],color="blue", s=60, label="DL")
        ax.scatter([],[],color="gray", s=60, label="Classical")
        ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig("outputs/plots/p4_catropy_scatter.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("  ✓ p4_catropy_scatter.png")
else:
    print("  ⚠ p4 skipped — catropy_df not available")

# ── P5: DL loss curves ──
dl_hist_keys = [k for k in dl_history if k in _dl_set]
if dl_hist_keys:
    cols_p=4; rows_p=(len(dl_hist_keys)+cols_p-1)//cols_p
    fig, axes = plt.subplots(rows_p, cols_p, figsize=(22, 5*rows_p))
    fig.suptitle("Train vs Validation Loss — DL Baselines\n"
                 "[AdamW | LR=0.001 | ReLU | Dropout=0.2 | Epochs=300 | EarlyStopping=OFF]",
                 fontsize=13, fontweight="bold")
    axes = axes.flatten()
    for idx, name in enumerate(dl_hist_keys):
        ax = axes[idx]; h = dl_history[name]
        ax.plot(h.get("loss",     []), "b-",  lw=2, label="Train")
        ax.plot(h.get("val_loss", []), "r--", lw=2, label="Val")
        ax.set_title(name, fontsize=10, fontweight="bold")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Joint MSE Loss")
        ax.legend(fontsize=8); ax.grid(alpha=0.4)
    for ax in axes[len(dl_hist_keys):]: ax.axis("off")
    plt.tight_layout()
    plt.savefig("outputs/plots/p5_dl_loss_curves.png", dpi=120, bbox_inches="tight")
    plt.close()
    print("  ✓ p5_dl_loss_curves.png")
else:
    print("  ⚠ p5 skipped — dl_history empty (run Section 7 first)")

# ── P5b: CB+MotionNN correction NN loss curves ──
cbnn_hist_keys = [k for k in dl_history if k.startswith("cbnn_")]
if cbnn_hist_keys:
    fig, axes = plt.subplots(1, len(H_LABELS), figsize=(22, 5))
    fig.suptitle("CB+MotionNN — Correction NN Loss per Horizon\n"
                 "[AdamW | LR=0.001 | clipnorm=1.0 | Epochs=300]",
                 fontsize=13, fontweight="bold")
    for ax_i, h_label in enumerate(H_LABELS):
        ax = axes[ax_i]
        key = f"cbnn_{h_label}"
        if key in dl_history:
            h = dl_history[key]
            ax.plot(h.get("loss",[]),     "b-",  lw=2, label="Train")
            ax.plot(h.get("val_loss",[]), "r--", lw=2, label="Val")
        ax.set_title(f"Correction NN — {h_label}", fontsize=10, fontweight="bold")
        ax.set_xlabel("Epoch"); ax.set_ylabel("MSE (km²)")
        ax.legend(fontsize=8); ax.grid(alpha=0.4)
    plt.tight_layout()
    plt.savefig("outputs/plots/p5b_cbnn_loss_curves.png", dpi=120, bbox_inches="tight")
    plt.close()
    print("  ✓ p5b_cbnn_loss_curves.png")
else:
    print("  ⚠ p5b skipped — cbnn loss history not available")

# ── P6: Map plots ──
# Use mh_test if available, else skip gracefully
if "mh_test" in dir() and len(mh_test) > 0:
    # Determine best ML and DL names safely
    ml_cands = sort_by_3h([n for n in ml_names
                            if "Persistence" not in n and "Stacking" not in n])
    dl_cands = sort_by_3h(dl_names)
    # Fall back to any available model if lists are empty
    all_available = sort_by_3h(list(all_results["3h"].keys()))
    best_ml_name = ml_cands[0] if ml_cands else \
                   (all_available[0] if all_available else None)
    best_dl_name = dl_cands[0]  if dl_cands  else None

    # Build panel list — only include models that have predictions
    panel_candidates = []
    if best_ml_name and best_ml_name in all_preds["3h"]:
        panel_candidates.append((f"Best ML ({best_ml_name})", "3h", best_ml_name))
    if best_dl_name and best_dl_name in all_preds["3h"]:
        panel_candidates.append((f"Best DL ({best_dl_name})", "3h", best_dl_name))
    for cm in ["CatBoost","CB+MotionNN","SECE Ensemble",
               "Persistence Residual Cascade","SECE+PRC Meta"]:
        if cm in all_preds["3h"]:
            panel_candidates.append((cm, "3h", cm))

    n_panels = min(6, len(panel_candidates))
    if n_panels > 0:
        panel_candidates = panel_candidates[:n_panels]
        cols_map = min(3, n_panels); rows_map = (n_panels+cols_map-1)//cols_map
        fig, axes = plt.subplots(rows_map, cols_map,
                                  figsize=(8*cols_map, 7*rows_map))
        axes = np.array(axes).flatten()
        fig.suptitle("True vs Predicted Tracks | 10 Test Storms | 3h Lead",
                     fontsize=14, fontweight="bold")

        top_storms = (mh_test.groupby("SID").size()
                      .sort_values(ascending=False).head(10).index.tolist())
        mh_sid = mh_test["SID"].values
        mh_lat = mh_test["LAT"].values
        mh_lon = mh_test["LON"].values
        n_mh   = len(mh_test)

        for ax_i, (panel_name, h_label, mname) in enumerate(panel_candidates):
            ax = axes[ax_i]; ax.set_facecolor("#d4eaf7")
            ax.set_xlim(70,100); ax.set_ylim(5,30)
            p_store = all_preds[h_label].get(mname)
            for s_idx, sid in enumerate(top_storms):
                sid_mask = mh_sid == sid
                if sid_mask.sum() == 0: continue
                true_lat = mh_lat[sid_mask]; true_lon = mh_lon[sid_mask]
                ax.plot(true_lon, true_lat, "-", lw=1.2, alpha=0.7,
                        color=TRACK_COLORS[s_idx%9])
                if p_store is None: continue
                n_p = len(p_store)
                if n_p == n_mh:
                    pred_lat = true_lat + p_store[sid_mask, 0]
                    pred_lon = true_lon + p_store[sid_mask, 1]
                    ax.plot(pred_lon, pred_lat, "--", lw=1, alpha=0.7,
                            color=TRACK_COLORS[s_idx%9])
                elif n_p < n_mh:
                    sid_indices = np.where(sid_mask)[0]
                    valid = sid_indices[sid_indices >= SEQ_LEN-1]
                    p_indices = valid - (SEQ_LEN-1)
                    in_range = p_indices < n_p
                    valid = valid[in_range]; p_indices = p_indices[in_range]
                    if len(p_indices) == 0: continue
                    pred_lat = mh_lat[valid] + p_store[p_indices, 0]
                    pred_lon = mh_lon[valid] + p_store[p_indices, 1]
                    ax.plot(pred_lon, pred_lat, "--", lw=1, alpha=0.7,
                            color=TRACK_COLORS[s_idx%9])
            ax.scatter([90.4],[23.7],c="red",s=150,marker="*",zorder=10)
            ax.set_title(panel_name, fontsize=10, fontweight="bold")
            ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
            ax.grid(alpha=0.3)
            ax.text(0.02,0.97,"─ True  -- Pred",transform=ax.transAxes,
                    fontsize=8,va="top")
        for ax in axes[n_panels:]: ax.axis("off")
        plt.tight_layout()
        plt.savefig("outputs/plots/p6_map_track_comparison.png",
                    dpi=130, bbox_inches="tight")
        plt.close()
        print("  ✓ p6_map_track_comparison.png")
    else:
        print("  ⚠ p6 skipped — no model predictions available in all_preds")
else:
    print("  ⚠ p6 skipped — mh_test not available (run Section 4 first)")

# ── P7: Bangladesh region overview ──
if "df_ts3" in dir() and len(df_ts3) > 0:
    fig, ax = plt.subplots(figsize=(11,10))
    ax.set_facecolor("#b0d4e8"); ax.set_xlim(70,100); ax.set_ylim(5,30)
    _test_storms = list(df_ts3["SID"].unique()[:50])
    for s_idx, sid in enumerate(_test_storms):
        sdf = df_ts3[df_ts3["SID"]==sid]
        if len(sdf)<3: continue
        ax.plot(sdf["LON"].values, sdf["LAT"].values, "-",
                lw=0.8, alpha=0.4, color=COLORS[s_idx%20])
    ax.scatter([90.4],[23.7],c="red",s=200,marker="*",zorder=10,label="Dhaka")
    ax.set_xlabel("Longitude",fontsize=12); ax.set_ylabel("Latitude",fontsize=12)
    ax.set_title("Bangladesh Region — Test Set Cyclone Tracks",
                 fontsize=13,fontweight="bold")
    ax.legend(fontsize=10); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("outputs/plots/p7_bangladesh_region_map.png",
                dpi=130, bbox_inches="tight")
    plt.close()
    print("  ✓ p7_bangladesh_region_map.png")
else:
    print("  ⚠ p7 skipped — df_ts3 not available")

# ── P8: Top-4 ML vs Top-4 DL catropy comparison ──
if "catropy_df" in dir() and len(catropy_df)>0 and top4_ml and top4_dl:
    fig, axes = plt.subplots(1,2,figsize=(16,6))
    fig.suptitle("Catropy: Top-4 ML vs Top-4 DL  (3h and 48h)",
                 fontsize=13,fontweight="bold")
    for ax_i, h_label in enumerate(["3h","48h"]):
        ax = axes[ax_i]
        sub    = catropy_df[catropy_df["horizon"]==h_label]
        ml_sub = sub[sub["model"].isin(top4_ml)].sort_values("catropy")
        dl_sub = sub[sub["model"].isin(top4_dl)].sort_values("catropy")
        all_sub= pd.concat([ml_sub, dl_sub])
        if len(all_sub)==0:
            ax.set_title(f"{h_label} — no data"); continue
        colors = ["steelblue"]*len(ml_sub)+["darkorange"]*len(dl_sub)
        bars = ax.barh(all_sub["model"], all_sub["catropy"],
                       color=colors, edgecolor="black", lw=0.5)
        ax.bar_label(bars,fmt="%.3f",padding=2,fontsize=8)
        ax.set_title(h_label,fontsize=12,fontweight="bold")
        ax.set_xlabel("Catropy Score"); ax.invert_yaxis(); ax.grid(axis="x",alpha=0.4)
        ax.legend(handles=[
            plt.Rectangle((0,0),1,1,fc="steelblue",label="ML"),
            plt.Rectangle((0,0),1,1,fc="darkorange",label="DL"),
        ],fontsize=9)
    plt.tight_layout()
    plt.savefig("outputs/plots/p8_catropy_ml_vs_dl.png",dpi=130,bbox_inches="tight")
    plt.close()
    print("  ✓ p8_catropy_ml_vs_dl.png")
else:
    print("  ⚠ p8 skipped — catropy_df or top4 lists not available")

print("\n  All plots saved to outputs/plots/")


12. CATROPY — TRAJECTORY ENTROPY / STABILITY
  ✓ catropy_scores_all_models.csv  (92 rows)

13. GENERATING ALL PLOTS
  ✓ p1_model_comparison_3h.png
  ✓ p2_error_vs_horizon.png
  ✓ p3_catropy_vs_horizon.png
  ✓ p4_catropy_scatter.png
  ✓ p5_dl_loss_curves.png
  ⚠ p5b skipped — cbnn loss history not available
  ✓ p6_map_track_comparison.png
  ✓ p7_bangladesh_region_map.png
  ✓ p8_catropy_ml_vs_dl.png

  All plots saved to outputs/plots/


In [ ]:
# ─────────────────────────────────────────────────────────────
# 14. ITERATIVE ROLLOUT (Mode B)
# Simulates true autoregressive prediction:
# Given a starting position, repeatedly predict 3h steps and
# chain them to reach 12h, 24h, and 48h positions.
# Models evaluated: LightGBM (best classical) + SECE-proxy
# Note: true SECE iterative rollout is approximated using the
# best individual base (LGB full-feature) since SECE is an
# ensemble that requires all 28 bases at each step.
# ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("14. ITERATIVE ROLLOUT SIMULATION")
print("="*60)

# ── Train LightGBM on 3h data (best classical for rollout) ──
lgb_sim_lat = lgb.LGBMRegressor(**LGB_P)
lgb_sim_lon = lgb.LGBMRegressor(**LGB_P)
lgb_sim_lat.fit(X3_tr, y3_tr[:, 0])
lgb_sim_lon.fit(X3_tr, y3_tr[:, 1])
print("  LightGBM (iterative) trained")

# ── Train XGBoost for second model comparison ──
xgb_sim_lat = xgb.XGBRegressor(**XGB_P)
xgb_sim_lon = xgb.XGBRegressor(**XGB_P)
xgb_sim_lat.fit(X3_tr, y3_tr[:, 0])
xgb_sim_lon.fit(X3_tr, y3_tr[:, 1])
print("  XGBoost (iterative) trained")

def simulate_rollout(mdl_lat, mdl_lon, model_name, storm_limit=None):
    """
    True iterative rollout: chain 3h predictions to reach 12h/24h/48h.
    Uses ALL test storms unless storm_limit is set.
    Returns a DataFrame of per-row predictions with haversine errors.
    """
    rows_iter = []
    sids = list(test_storms)
    if storm_limit:
        sids = sids[:storm_limit]

    for sid in sids:
        sdf = df[df["SID"] == sid].sort_values("ISO_TIME").reset_index(drop=True)
        max_steps = max(H_STEPS)  # 16
        if len(sdf) < max_steps + 2:
            continue

        for t in range(len(sdf) - max_steps):
            row0 = sdf.iloc[t]
            cur_lat = float(row0["LAT"])
            cur_lon = float(row0["LON"])

            # Initialize lag windows
            lag_l  = [float(sdf.iloc[max(0, t-j)]["LAT"]) for j in range(1, 4)]
            lag_n  = [float(sdf.iloc[max(0, t-j)]["LON"]) for j in range(1, 4)]
            lag_dl = [float(sdf.iloc[max(0, t-j)]["dLAT"])
                      if "dLAT" in sdf.columns else 0.0 for j in range(1, 4)]
            lag_dn = [float(sdf.iloc[max(0, t-j)]["dLON"])
                      if "dLON" in sdf.columns else 0.0 for j in range(1, 4)]

            # Step through 3h increments up to max_steps
            positions = [(cur_lat, cur_lon)]  # store positions at each 3h step
            ll = lag_l[:]; ln = lag_n[:]; ldl = lag_dl[:]; ldn = lag_dn[:]

            for step in range(max_steps):
                fr = row0.copy()
                fr["LAT"] = cur_lat; fr["LON"] = cur_lon
                fr["LAT_lag1"] = ll[0]; fr["LON_lag1"] = ln[0]
                fr["dLAT_lag1"] = ldl[0]; fr["dLON_lag1"] = ldn[0]
                fr["LAT_lag2"] = ll[1]; fr["LON_lag2"] = ln[1]
                fr["dLAT_lag2"] = ldl[1]; fr["dLON_lag2"] = ldn[1]
                fr["LAT_lag3"] = ll[2]; fr["LON_lag3"] = ln[2]
                fr["dLAT_lag3"] = ldl[2]; fr["dLON_lag3"] = ldn[2]

                fv = np.array([[fr.get(c, 0.0) if hasattr(fr, 'get')
                                else (fr[c] if c in fr.index else 0.0)
                                for c in ALL_FEATURE_COLS]])
                fv = np.nan_to_num(fv, nan=0.0)
                fs = scaler.transform(imputer.transform(fv))

                d_lat = float(mdl_lat.predict(fs)[0])
                d_lon = float(mdl_lon.predict(fs)[0])
                cur_lat += d_lat; cur_lon += d_lon
                ll = [cur_lat - d_lat] + ll[:2]
                ln = [cur_lon - d_lon] + ln[:2]
                ldl = [d_lat] + ldl[:2]
                ldn = [d_lon] + ldn[:2]
                positions.append((cur_lat, cur_lon))

            # Record error at each target horizon (12h=step4, 24h=step8, 48h=step16)
            for step_k, h_label in zip(H_STEPS[1:], H_LABELS[1:]):
                if t + step_k >= len(sdf):
                    continue
                pred_lat, pred_lon = positions[step_k]
                fut = sdf.iloc[t + step_k]
                err = haversine_km(pred_lat, pred_lon,
                                   float(fut["LAT"]), float(fut["LON"]))
                rows_iter.append({
                    "model":      model_name,
                    "SID":        sid,
                    "t":          t,
                    "horizon":    h_label,
                    "pred_lat":   pred_lat,
                    "pred_lon":   pred_lon,
                    "true_lat":   float(fut["LAT"]),
                    "true_lon":   float(fut["LON"]),
                    "haversine_km": err
                })

    return pd.DataFrame(rows_iter)

print("\n  Running iterative simulation — LightGBM (all test storms)...")
iter_lgb = simulate_rollout(lgb_sim_lat, lgb_sim_lon, "LightGBM (iterative)")

print(f"  Running iterative simulation — XGBoost (all test storms)...")
iter_xgb = simulate_rollout(xgb_sim_lat, xgb_sim_lon, "XGBoost (iterative)")

iter_all = pd.concat([iter_lgb, iter_xgb], ignore_index=True)

# Summary table
iter_summary = iter_all.groupby(["model", "horizon"])["haversine_km"].agg(
    median="median",
    mean="mean",
    p75=lambda x: np.percentile(x, 75),
    p90=lambda x: np.percentile(x, 90)
).round(2).reset_index()
iter_summary.columns = ["model", "horizon", "median_km", "mean_km", "p75_km", "p90_km"]

iter_summary.to_csv(
    "outputs/metrics/metrics_rollout_iterative_standardized.csv", index=False)
print("  ✓ metrics_rollout_iterative_standardized.csv")
print(iter_summary.to_string(index=False))

# Quick comparison: direct vs iterative at same horizons
print("\n  Direct (multi-horizon) vs Iterative rollout comparison:")
print(f"  {'Horizon':<8} {'LGB Direct':>12} {'LGB Iterative':>14}")
for h in H_LABELS[1:]:
    direct = all_results[h].get("LightGBM", {}).get("median_km", float("nan"))
    sub = iter_summary[(iter_summary["model"]=="LightGBM (iterative)") &
                       (iter_summary["horizon"]==h)]
    it_val = sub["median_km"].values[0] if len(sub) > 0 else float("nan")
    print(f"  {h:<8} {direct:>12.2f} {it_val:>14.2f}")



14. ITERATIVE ROLLOUT SIMULATION
  LightGBM (iterative) trained
  XGBoost (iterative) trained

  Running iterative simulation — LightGBM (all test storms)...
  Running iterative simulation — XGBoost (all test storms)...
  ✓ metrics_rollout_iterative_standardized.csv
               model horizon  median_km  mean_km  p75_km  p90_km
LightGBM (iterative)     12h      29.60    39.31   50.59   79.46
LightGBM (iterative)     24h      91.02   116.42  155.22  233.22
LightGBM (iterative)     48h     276.93   323.29  427.63  628.25
 XGBoost (iterative)     12h      29.45    39.28   51.13   80.18
 XGBoost (iterative)     24h      90.45   115.52  153.61  235.20
 XGBoost (iterative)     48h     271.91   320.89  427.97  622.32

  Direct (multi-horizon) vs Iterative rollout comparison:
  Horizon    LGB Direct  LGB Iterative
  12h             30.80          29.60
  24h             88.12          91.02
  48h            227.94         276.93


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 15 — SATELLITE TRAJECTORY MAPS  (REWRITTEN — all fixes applied)
# Run AFTER Cell 1 completes. Kaggle: Settings → Internet ON
#
# FIXES APPLIED:
#   1. get_track: custom models at 3h use mh_test (not df_ts3)
#   2. auto_extent: fixed full BoB [80,97,7,24] — Bangladesh always visible
#   3. All 7 figures regenerated cleanly
# ══════════════════════════════════════════════════════════════════════════════

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "cartopy", "-q"],
               capture_output=True)

import os, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
warnings.filterwarnings("ignore")

import cartopy.crs as ccrs
import cartopy.io.img_tiles as cimgt
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

os.makedirs("outputs/trajectory_maps", exist_ok=True)
print("Libraries loaded ✓")

# ─────────────────────────────────────────────────────────────
# 1. CONSTANTS
# ─────────────────────────────────────────────────────────────
SATELLITE  = cimgt.QuadtreeTiles()   # Bing satellite imagery
TILE_ZOOM  = 6                        # 6 = good quality + fast. Use 7 for final print.
PROJ       = ccrs.PlateCarree()

# Fixed Bay of Bengal extent — every BoB cyclone fits in this box,
# Bangladesh coast always visible at top, consistent scale across all figures
BOB_EXTENT = [80, 97, 7, 24]         # [lon_min, lon_max, lat_min, lat_max]

# Gridline ticks — fixed to match BOB_EXTENT
BOB_LON_TICKS = [81, 84, 87, 90, 93, 96]
BOB_LAT_TICKS = [9, 12, 15, 18, 21, 24]

# ─────────────────────────────────────────────────────────────
# 2. TRACK STYLES
# ─────────────────────────────────────────────────────────────
STYLES = {
    "Actual (IBTrACS)":             {"color": "#FFFFFF", "lw": 2.2, "ms": 6,  "ls": "-",  "marker": "o", "zorder": 9},
    "SECE Ensemble":                {"color": "#FF2020", "lw": 2.2, "ms": 5,  "ls": "-",  "marker": "o", "zorder": 8},
    "Persistence Residual Cascade": {"color": "#FFD700", "lw": 1.8, "ms": 4,  "ls": "-",  "marker": "o", "zorder": 7},
    "CB+MotionNN":                  {"color": "#44AAFF", "lw": 1.8, "ms": 4,  "ls": "-",  "marker": "o", "zorder": 6},
    "LightGBM":                     {"color": "#00E676", "lw": 1.6, "ms": 3,  "ls": "--", "marker": "s", "zorder": 5},
    "XGBoost":                      {"color": "#FF9800", "lw": 1.6, "ms": 3,  "ls": "--", "marker": "s", "zorder": 5},
    "Random Forest":                {"color": "#CE93D8", "lw": 1.6, "ms": 3,  "ls": "--", "marker": "s", "zorder": 5},
    "Stacking Ensemble":            {"color": "#80DEEA", "lw": 1.6, "ms": 3,  "ls": "--", "marker": "s", "zorder": 5},
    "BLSTM":                        {"color": "#F48FB1", "lw": 1.4, "ms": 3,  "ls": ":",  "marker": "^", "zorder": 4},
    "CNN-GRU":                      {"color": "#BCAAA4", "lw": 1.4, "ms": 3,  "ls": ":",  "marker": "^", "zorder": 4},
    "GRU":                          {"color": "#EF9A9A", "lw": 1.4, "ms": 3,  "ls": ":",  "marker": "^", "zorder": 4},
}

CUSTOM_MODELS     = ["SECE Ensemble", "Persistence Residual Cascade", "CB+MotionNN"]
CUSTOM_MODELS_SET = set(CUSTOM_MODELS)
DL_MODELS         = ["BLSTM", "CNN-GRU", "CNN-LSTM", "GRU", "LSTM",
                      "GRU-LSTM", "CNN", "SLSTM", "Hybrid Transformer"]
ML_MODELS         = ["XGBoost", "LightGBM", "Random Forest", "Gradient Boosting",
                      "HistGradientBoosting", "CatBoost", "Stacking Ensemble"]

# ─────────────────────────────────────────────────────────────
# 3. STORM NAME LOOKUP
# ─────────────────────────────────────────────────────────────
_DATA_PATH = ("/kaggle/input/datasets/anybodyk/bbddddd/bangladesh_nextstep_dataset_research_ready.csv")
_raw = pd.read_csv(_DATA_PATH, usecols=["SID", "NAME", "SEASON"])
_raw = _raw.drop_duplicates("SID").set_index("SID")

def storm_label(sid):
    if sid in _raw.index:
        nm = str(_raw.loc[sid, "NAME"])
        yr = int(_raw.loc[sid, "SEASON"])
        return f"{nm.title()} ({yr})" if nm.upper() != "UNNAMED" else f"Unnamed ({yr})"
    return sid

# ─────────────────────────────────────────────────────────────
# 4. HELPERS
# ─────────────────────────────────────────────────────────────
def get_style(model_name):
    return STYLES.get(model_name,
                      {"color": "#AAAAAA", "lw": 1.4, "ms": 3,
                       "ls": "--", "marker": "o", "zorder": 4})

def get_track(sid, h_label, model_name):
    """
    Returns (pred_lats, pred_lons) for a model on a storm.

    KEY FIX:
      - Custom models at 3h use mh_test + dLAT_3h/dLON_3h columns
        (all_preds["3h"] is indexed against mh_test, not df_ts3)
      - Baseline models at 3h use df_ts3 + dLAT/dLON columns
      - All models at 12h/24h/48h use mh_test
      - DL models handled by clipping idx to len(pred_arr)
    """
    if h_label == "3h" and model_name in CUSTOM_MODELS_SET:
        # Custom models stored predictions on mh_test rows
        frame    = mh_test.reset_index(drop=True)
        dlat_col = "dLAT_3h"
        dlon_col = "dLON_3h"
    elif h_label == "3h":
        # Baseline models stored predictions on df_ts3 rows
        frame    = df_ts3.reset_index(drop=True)
        dlat_col = "dLAT"
        dlon_col = "dLON"
    else:
        frame    = mh_test.reset_index(drop=True)
        dlat_col = f"dLAT_{h_label}"
        dlon_col = f"dLON_{h_label}"

    idx = np.where((frame["SID"] == sid).values)[0]
    if len(idx) == 0:
        return None, None

    lat0 = frame.loc[idx, "LAT"].values
    lon0 = frame.loc[idx, "LON"].values

    if model_name == "Actual (IBTrACS)":
        return (lat0 + frame.loc[idx, dlat_col].values,
                lon0 + frame.loc[idx, dlon_col].values)

    if model_name not in all_preds.get(h_label, {}):
        return None, None

    pred_arr  = all_preds[h_label][model_name]
    valid_idx = idx[idx < len(pred_arr)]
    if len(valid_idx) == 0:
        return None, None

    return (frame.loc[valid_idx, "LAT"].values + pred_arr[valid_idx, 0],
            frame.loc[valid_idx, "LON"].values + pred_arr[valid_idx, 1])

def auto_extent(sid=None, pad=1.0):
    """
    Fixed full Bay of Bengal extent.
    Every BoB cyclone fits within this box.
    Bangladesh coast always visible at top.
    Consistent scale across ALL figures in the paper.
    """
    return BOB_EXTENT

def best_n(model_list, h_label, n=3):
    """Return top-n models from list by median_km at h_label."""
    ranked = sorted(
        [(m, all_results[h_label][m]["median_km"])
         for m in model_list if m in all_results.get(h_label, {})],
        key=lambda x: x[1]
    )
    return [r[0] for r in ranked[:n]]

# ─────────────────────────────────────────────────────────────
# 5. CORE PANEL DRAWING FUNCTION
# ─────────────────────────────────────────────────────────────
def draw_panel(ax, sid, h_label, model_names,
               panel_letter="a", title=""):
    """
    Draw ONE satellite map panel with storm tracks.
    Fixed BoB extent — Bangladesh always in frame.
    """
    ax.set_extent(BOB_EXTENT, crs=PROJ)

    # ── Bing satellite basemap ─────────────────────────────
    ax.add_image(SATELLITE, TILE_ZOOM)

    # ── Fixed gridlines ────────────────────────────────────
    gl = ax.gridlines(crs=PROJ, draw_labels=True,
                      linewidth=0.5, color="white",
                      alpha=0.6, linestyle="--", zorder=3)
    gl.top_labels   = False
    gl.right_labels = False
    gl.xformatter   = LONGITUDE_FORMATTER
    gl.yformatter   = LATITUDE_FORMATTER
    gl.xlocator     = mticker.FixedLocator(BOB_LON_TICKS)
    gl.ylocator     = mticker.FixedLocator(BOB_LAT_TICKS)
    gl.xlabel_style = {"size": 12, "color": "black", "weight": "bold"}
    gl.ylabel_style = {"size": 12, "color": "black", "weight": "bold"}

    # ── Model tracks ──────────────────────────────────────
    legend_handles = []
    for mname in model_names:
        st   = get_style(mname)
        lats, lons = get_track(sid, h_label, mname)
        if lats is None or len(lats) < 2:
            continue

        c  = st["color"]
        lw = st["lw"]
        ms = st["ms"]
        zo = st["zorder"]

        # White halo — track visible on any terrain/ocean background
        ax.plot(lons, lats, color="white", lw=lw + 1.8,
                alpha=0.30, zorder=zo - 1, transform=PROJ)

        # Main track line
        ax.plot(lons, lats, color=c, lw=lw,
                ls=st["ls"], marker=st["marker"],
                ms=ms, markerfacecolor=c,
                markeredgecolor="white", markeredgewidth=0.45,
                transform=PROJ, zorder=zo)

        # Larger start dot
        ax.plot(lons[0], lats[0], "o", color=c, ms=ms + 4,
                markeredgecolor="white", markeredgewidth=1.0,
                transform=PROJ, zorder=zo + 1)

        # Direction arrow at track end
        if len(lons) > 1:
            ax.annotate(
                "",
                xy=(lons[-1], lats[-1]),
                xytext=(lons[-2], lats[-2]),
                xycoords=PROJ._as_mpl_transform(ax),
                textcoords=PROJ._as_mpl_transform(ax),
                arrowprops=dict(arrowstyle="->", color=c,
                                lw=1.8, mutation_scale=16),
                zorder=zo + 2
            )

        # Legend label — show median km for custom models
        med_str = ""
        if mname in CUSTOM_MODELS_SET and mname in all_results.get(h_label, {}):
            med_str = f"  ({all_results[h_label][mname]['median_km']:.1f} km)"
        legend_handles.append(
            mlines.Line2D([], [], color=c, lw=lw, ls=st["ls"],
                          marker=st["marker"], ms=ms,
                          markeredgecolor="white", markeredgewidth=0.45,
                          label=f"{mname}{med_str}"))

    # ── Legend ────────────────────────────────────────────
    ax.legend(handles=legend_handles,
              loc="lower left", fontsize=8.5,
              framealpha=0.92, edgecolor="#444444",
              facecolor="white", handlelength=2.5,
              borderpad=0.6)

    # ── North arrow ───────────────────────────────────────
    ax.text(0.93, 0.10, "N",
            transform=ax.transAxes, color="black",
            fontweight="bold", fontsize=16,
            ha="center", va="center",
            bbox=dict(facecolor="white", edgecolor="black", pad=5),
            zorder=12)
    ax.annotate("", xy=(0.93, 0.15), xytext=(0.93, 0.12),
                xycoords="axes fraction",
                arrowprops=dict(facecolor="black", width=2, headwidth=8),
                zorder=12)

    # ── Scale bar (~150 km) ───────────────────────────────
    scale_km  = 150
    scale_deg = scale_km / 111.0
    sl = BOB_EXTENT[0] + 0.4
    sb = BOB_EXTENT[2] + 0.4
    ax.plot([sl, sl + scale_deg], [sb, sb],
            "w-", lw=2.5, zorder=10, transform=PROJ)
    ax.plot([sl] * 2, [sb - 0.12, sb + 0.12],
            "w-", lw=2.0, zorder=10, transform=PROJ)
    ax.plot([sl + scale_deg] * 2, [sb - 0.12, sb + 0.12],
            "w-", lw=2.0, zorder=10, transform=PROJ)
    ax.text(sl + scale_deg / 2, sb - 0.5, f"{scale_km} km",
            ha="center", fontsize=8, fontweight="bold", color="white",
            transform=PROJ, zorder=10,
            bbox=dict(boxstyle="round,pad=0.1",
                      fc="black", ec="none", alpha=0.55))

    # ── Panel title ───────────────────────────────────────
    display_title = title if title else f"{panel_letter}. {storm_label(sid)}"
    ax.set_title(display_title,
                 fontweight="bold", fontsize=20, loc="left",
                 pad=10, color="white",
                 bbox=dict(facecolor="black", alpha=0.5, edgecolor="none"))

# ─────────────────────────────────────────────────────────────
# 6. PICK TARGET STORMS
# ─────────────────────────────────────────────────────────────
named_sids = sorted(
    [s for s in test_storms
     if s in _raw.index
     and "UNNAMED" not in str(_raw.loc[s, "NAME"]).upper()],
    key=lambda s: int(_raw.loc[s, "SEASON"]),
    reverse=True
)[:6]

print(f"\nTarget storms ({len(named_sids)}):")
for s in named_sids:
    print(f"  {s}  →  {storm_label(s)}")

# ─────────────────────────────────────────────────────────────
# F1. PER-CYCLONE 3-PANEL (3h / 12h / 24h)
#     Custom models vs Actual — one figure per storm
#     SECE now shows at 3h (fixed get_track)
# ─────────────────────────────────────────────────────────────
print("\n[F1] Per-cyclone 3-panel (3h / 12h / 24h) ...")
for sid in named_sids:
    lbl = storm_label(sid)
    fig, axes = plt.subplots(1, 3, figsize=(24, 9),
                             subplot_kw={"projection": PROJ},
                             facecolor="white")
    fig.suptitle(f"Trajectory Validation — Cyclone {lbl}\n"
                 f"Proposed Models vs Actual IBTrACS Track",
                 fontsize=14, fontweight="bold", y=1.01)
    fig.subplots_adjust(wspace=0.06)

    for ax, h, letter in zip(axes, ["3h", "12h", "24h"], ["a", "b", "c"]):
        draw_panel(ax, sid, h,
                   model_names=["Actual (IBTrACS)"] + CUSTOM_MODELS,
                   panel_letter=letter,
                   title=f"{letter}. {h} Lead Time")

    safe = lbl.replace(" ", "_").replace("(", "").replace(")", "")
    path = f"outputs/trajectory_maps/F1_{safe}_custom_vs_actual.png"
    fig.savefig(path, dpi=200, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  Saved: {path}")

# ─────────────────────────────────────────────────────────────
# F2. DUAL PANEL — 2 storms at 24h (exact Sidr/Aila style)
# ─────────────────────────────────────────────────────────────
print("\n[F2] Dual panel (Sidr/Aila style) ...")
if len(named_sids) >= 2:
    for pair_start in range(0, min(len(named_sids) - 1, 4), 2):
        sid1 = named_sids[pair_start]
        sid2 = named_sids[pair_start + 1]

        fig, axes = plt.subplots(1, 2, figsize=(20, 10),
                                 subplot_kw={"projection": PROJ},
                                 facecolor="white")
        fig.subplots_adjust(wspace=0.08)

        draw_panel(axes[0], sid1, "24h",
                   model_names=["Actual (IBTrACS)"] + CUSTOM_MODELS,
                   panel_letter="a",
                   title=f"a. {storm_label(sid1)}")
        draw_panel(axes[1], sid2, "24h",
                   model_names=["Actual (IBTrACS)"] + CUSTOM_MODELS,
                   panel_letter="b",
                   title=f"b. {storm_label(sid2)}")

        lbl1 = storm_label(sid1).replace(" ", "_").replace("(","").replace(")","")
        lbl2 = storm_label(sid2).replace(" ", "_").replace("(","").replace(")","")
        path = f"outputs/trajectory_maps/F2_dual_{lbl1}_{lbl2}_24h.png"
        fig.savefig(path, dpi=200, bbox_inches="tight", facecolor="white")
        plt.close(fig)
        print(f"  Saved: {path}")

# ─────────────────────────────────────────────────────────────
# F3. HEAD-TO-HEAD — Best ML vs Best DL vs SECE across 3 horizons
# ─────────────────────────────────────────────────────────────
print("\n[F3] Head-to-head across 3 horizons ...")
best_ml_list = best_n([m for m in ML_MODELS if "Persistence" not in m], "24h", 1)
best_dl_list = best_n(DL_MODELS, "24h", 1)
best_ml_name = best_ml_list[0] if best_ml_list else "LightGBM"
best_dl_name = best_dl_list[0] if best_dl_list else "BLSTM"

for sid in named_sids[:3]:
    lbl = storm_label(sid)
    fig, axes = plt.subplots(1, 3, figsize=(24, 9),
                             subplot_kw={"projection": PROJ},
                             facecolor="white")
    fig.suptitle(
        f"Head-to-Head — Cyclone {lbl}\n"
        f"Best ML ({best_ml_name}) vs Best DL ({best_dl_name}) vs SECE Ensemble",
        fontsize=13, fontweight="bold", y=1.01)
    fig.subplots_adjust(wspace=0.06)

    for ax, h, letter in zip(axes, ["3h", "12h", "24h"], ["a", "b", "c"]):
        draw_panel(ax, sid, h,
                   model_names=["Actual (IBTrACS)",
                                best_ml_name,
                                best_dl_name,
                                "SECE Ensemble"],
                   panel_letter=letter,
                   title=f"{letter}. {h} Lead Time")

    safe = lbl.replace(" ", "_").replace("(","").replace(")","")
    path = f"outputs/trajectory_maps/F3_{safe}_headtohead.png"
    fig.savefig(path, dpi=200, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  Saved: {path}")

# ─────────────────────────────────────────────────────────────
# F4. TOP-3 DL vs TOP-3 ML — side by side at 24h
# ─────────────────────────────────────────────────────────────
print("\n[F4] DL vs ML comparison at 24h ...")
top3_dl = best_n(DL_MODELS, "24h", 3)
top3_ml = best_n(ML_MODELS, "24h", 3)

for sid in named_sids[:2]:
    lbl = storm_label(sid)
    fig, axes = plt.subplots(1, 2, figsize=(20, 9),
                             subplot_kw={"projection": PROJ},
                             facecolor="white")
    fig.suptitle(f"Deep Learning vs Classical ML — Cyclone {lbl} (24h)",
                 fontsize=13, fontweight="bold", y=1.01)
    fig.subplots_adjust(wspace=0.08)

    draw_panel(axes[0], sid, "24h",
               model_names=["Actual (IBTrACS)"] + top3_dl,
               panel_letter="a",
               title="a. Top-3 Deep Learning Models")
    draw_panel(axes[1], sid, "24h",
               model_names=["Actual (IBTrACS)"] + top3_ml,
               panel_letter="b",
               title="b. Top-3 Classical ML Models")

    safe = lbl.replace(" ", "_").replace("(","").replace(")","")
    path = f"outputs/trajectory_maps/F4_{safe}_DL_vs_ML_24h.png"
    fig.savefig(path, dpi=200, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  Saved: {path}")

# ─────────────────────────────────────────────────────────────
# F5. ALL HORIZONS GRID — 3 models × 3 horizons (9 panels)
# ─────────────────────────────────────────────────────────────
print("\n[F5] All-horizons grid (3 models × 3 horizons) ...")
for sid in named_sids[:2]:
    lbl      = storm_label(sid)
    horizons = ["3h", "12h", "24h"]

    fig, axes = plt.subplots(3, 3, figsize=(24, 27),
                             subplot_kw={"projection": PROJ},
                             facecolor="white")
    fig.suptitle(f"Multi-Horizon Track Validation — Cyclone {lbl}",
                 fontsize=16, fontweight="bold", y=1.005)
    fig.subplots_adjust(wspace=0.06, hspace=0.10)

    letters = ["a","b","c","d","e","f","g","h","i"]
    for row, mname in enumerate(CUSTOM_MODELS):
        for col, h in enumerate(horizons):
            ax  = axes[row][col]
            let = letters[row * 3 + col]
            med = all_results.get(h, {}).get(mname, {}).get("median_km", 0)
            draw_panel(ax, sid, h,
                       model_names=["Actual (IBTrACS)", mname],
                       panel_letter=let,
                       title=f"{let}. {mname} — {h}  ({med:.1f} km)")

    safe = lbl.replace(" ", "_").replace("(","").replace(")","")
    path = f"outputs/trajectory_maps/F5_{safe}_3models_3horizons.png"
    fig.savefig(path, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  Saved: {path}")

# ─────────────────────────────────────────────────────────────
# F6. SUMMARY GRID — all named storms at 24h
# ─────────────────────────────────────────────────────────────
print("\n[F6] Summary grid — all named storms at 24h ...")
ncols     = 2
nrows     = int(np.ceil(len(named_sids) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(20, 10 * nrows),
                          subplot_kw={"projection": PROJ},
                          facecolor="white")
axes_flat = np.array(axes).flatten()
letters   = ["a","b","c","d","e","f","g","h"]

fig.suptitle("24h Track Prediction — Named Cyclones in Test Set\n"
             "SECE (Red) vs PRC (Yellow) vs CB+MotionNN (Blue) vs Actual (White)",
             fontsize=14, fontweight="bold")

for i, sid in enumerate(named_sids):
    draw_panel(axes_flat[i], sid, "24h",
               model_names=["Actual (IBTrACS)"] + CUSTOM_MODELS,
               panel_letter=letters[i],
               title=f"{letters[i]}. {storm_label(sid)}")

for j in range(len(named_sids), len(axes_flat)):
    axes_flat[j].set_visible(False)

path = "outputs/trajectory_maps/F6_summary_all_storms_24h.png"
fig.savefig(path, dpi=200, bbox_inches="tight", facecolor="white")
plt.close(fig)
print(f"  Saved: {path}")

# ─────────────────────────────────────────────────────────────
# F7. SECE RANK PROOF — bar chart (3h / 12h / 24h)
# ─────────────────────────────────────────────────────────────
print("\n[F7] SECE rank proof bar chart ...")
fig, axes = plt.subplots(1, 3, figsize=(21, 7), facecolor="#0d1b2a")
fig.suptitle("SECE Ensemble — Rank #1 at 3h, 12h, 24h\n"
             "Median Haversine Error (km) — Lower is Better",
             fontsize=14, fontweight="bold", color="white")

for ax, h in zip(axes, ["3h", "12h", "24h"]):
    ax.set_facecolor("#0d1b2a")
    sorted_models = sorted(
        [(m, all_results[h][m]["median_km"])
         for m in all_results[h]
         if m != "Persistence Baseline"],
        key=lambda x: x[1]
    )[:14]

    names  = [r[0] for r in sorted_models]
    values = [r[1] for r in sorted_models]
    colors = ["#FF2020" if n == "SECE Ensemble"
              else "#FFD700" if n in CUSTOM_MODELS_SET
              else "#4488AA"
              for n in names]

    bars = ax.barh(names, values, color=colors,
                   edgecolor="white", linewidth=0.4)
    ax.bar_label(bars, fmt="%.2f", padding=3,
                 fontsize=7.5, color="white")
    ax.set_xlabel("Median Haversine (km)", color="white", fontsize=10)
    ax.set_title(f"{h} Horizon", fontsize=12,
                 fontweight="bold", color="white")
    ax.invert_yaxis()
    ax.tick_params(colors="white", labelsize=8)
    for spine in ["bottom", "left"]:
        ax.spines[spine].set_color("white")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    if "SECE Ensemble" in names:
        idx_s = names.index("SECE Ensemble")
        ax.text(values[idx_s] / 2, idx_s, "★ RANK #1",
                ha="center", va="center",
                fontsize=8, fontweight="bold", color="white")

fig.legend(
    handles=[
        mpatches.Patch(color="#FF2020", label="SECE (Proposed)"),
        mpatches.Patch(color="#FFD700", label="Other Custom"),
        mpatches.Patch(color="#4488AA", label="Baseline"),
    ],
    loc="lower center", ncol=3, fontsize=10,
    facecolor="#0d1b2a", labelcolor="white",
    edgecolor="#444444", bbox_to_anchor=(0.5, -0.04)
)

plt.tight_layout()
path = "outputs/trajectory_maps/F7_SECE_rank_proof.png"
fig.savefig(path, dpi=200, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.close(fig)
print(f"  Saved: {path}")

# ─────────────────────────────────────────────────────────────
# DOWNLOAD ZIP
# ─────────────────────────────────────────────────────────────
import shutil
from IPython.display import FileLink, display as ipy_display

shutil.make_archive("trajectory_maps", "zip", "outputs/trajectory_maps")
print("\nDownload all figures:")
ipy_display(FileLink("trajectory_maps.zip"))

# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("Section 15 complete.")
print("Figures saved to: outputs/trajectory_maps/")
print("  F1_*_custom_vs_actual.png        3-panel per storm (3h/12h/24h)")
print("  F2_dual_*_24h.png                Sidr/Aila style dual panel")
print("  F3_*_headtohead.png              Best ML/DL/SECE 3 horizons")
print("  F4_*_DL_vs_ML_24h.png            Top-3 DL vs Top-3 ML")
print("  F5_*_3models_3horizons.png       9-panel full grid")
print("  F6_summary_all_storms_24h.png    All named storms")
print("  F7_SECE_rank_proof.png           Rank bar chart proof")
print("=" * 60)

Libraries loaded ✓

Target storms (6):
  2023317N10094  →  Midhili (2023)
  2023334N08088  →  Michaung (2023)
  2020335N06090  →  Burevi (2020)
  2018281N14088  →  Titli (2018)
  2012301N11087  →  Nilam (2012)
  2009344N06085  →  Ward (2009)

[F1] Per-cyclone 3-panel (3h / 12h / 24h) ...
  Saved: outputs/trajectory_maps/F1_Midhili_2023_custom_vs_actual.png
  Saved: outputs/trajectory_maps/F1_Michaung_2023_custom_vs_actual.png
  Saved: outputs/trajectory_maps/F1_Burevi_2020_custom_vs_actual.png
  Saved: outputs/trajectory_maps/F1_Titli_2018_custom_vs_actual.png
  Saved: outputs/trajectory_maps/F1_Nilam_2012_custom_vs_actual.png
  Saved: outputs/trajectory_maps/F1_Ward_2009_custom_vs_actual.png

[F2] Dual panel (Sidr/Aila style) ...
  Saved: outputs/trajectory_maps/F2_dual_Midhili_2023_Michaung_2023_24h.png
  Saved: outputs/trajectory_maps/F2_dual_Burevi_2020_Titli_2018_24h.png

[F3] Head-to-head across 3 horizons ...
  Saved: outputs/trajectory_maps/F3_Midhili_2023_headtohead.png
  Save

/kaggle/working/trajectory_maps.zip


Section 15 complete.
Figures saved to: outputs/trajectory_maps/
  F1_*_custom_vs_actual.png        3-panel per storm (3h/12h/24h)
  F2_dual_*_24h.png                Sidr/Aila style dual panel
  F3_*_headtohead.png              Best ML/DL/SECE 3 horizons
  F4_*_DL_vs_ML_24h.png            Top-3 DL vs Top-3 ML
  F5_*_3models_3horizons.png       9-panel full grid
  F6_summary_all_storms_24h.png    All named storms
  F7_SECE_rank_proof.png           Rank bar chart proof
